# 04 — Single Newscast OCR Analysis

**Parte 2 — Single Video Analysis / Newscast**

Este notebook usa o OCR de **um telejornal específico** para construir uma análise temporal baseada em três dimensões:

1. **Texto** — temas, entidades, palavras e frases visíveis no ecrã.
2. **Confiança** — qualidade/fiabilidade das deteções OCR ao longo do vídeo.
3. **Localização** — onde o texto aparece no ecrã, para distinguir rodapés, títulos, gráficos e possíveis mudanças de layout.

## Decisão metodológica desta versão

Para **segmentar notícias/blocos**, o notebook usa principalmente o OCR da zona inferior do ecrã, isto é, a região onde normalmente aparecem **títulos, subtítulos, rodapés e lower-thirds**.

- `ocr_all` mantém o OCR completo, útil para contexto, análise de qualidade e validação.
- `ocr_for_segmentation` usa apenas OCR que intersecta a zona inferior (`y2 >= 480` em vídeo 720p), e é a fonte usada para calcular temas, mudanças lexicais, transições e blocos.

Objetivo principal:

> usar OCR do rodapé/lower-third para construir uma timeline temática, detetar pontos candidatos a transição e estimar blocos/notícias, mantendo o OCR completo como contexto e validação.


## 0. Configuração

Edita apenas esta célula quando mudares de telejornal.

Estrutura esperada, igual à usada na parte 1:

```text
project_root/
├── data/
│   └── features/
│       ├── *_ocr.pkl
│       └── candidate_party.pkl  # opcional
└── outputs_ocr_04_single_newscast/
```


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import re
import warnings
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
FEATURES_DIR = DATA_DIR / "features"

# Root folder for OCR outputs from multiple newscasts.
OCR_OUTPUT_ROOT = BASE_DIR / "outputs_ocr_by_video"
OCR_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Escolha do telejornal
# ------------------------------------------------------------
TARGET_FILE = "Telejornal_RTP_Jan_13_ocr.pkl"

# Create a video id from the filename.
# Example:
# Telejornal_RTP_Nov_20_ocr.pkl -> Telejornal_RTP_Nov_20
VIDEO_ID = Path(TARGET_FILE).stem.replace("_ocr", "")

# Video-specific OCR output folder.
OUTPUT_DIR = OCR_OUTPUT_ROOT / VIDEO_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("TARGET_FILE:", TARGET_FILE)
print("VIDEO_ID:", VIDEO_ID)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

# Frames amostrados a 1 FPS no enunciado.
SAMPLE_RATE_FPS = 1

# Janelas temporais:
# - 3s permite detetar micro-mudanças no OCR.
# - Os blocos finais devem ser consolidados, porque micro-blocos de 3s não são necessariamente notícias.
WINDOW_SIZE_SECONDS = 3
MINUTE_SIZE_SECONDS = 60

# Filtro de confiança.
# Se o ficheiro quase não tiver confidence válida, o notebook não aplica filtro automaticamente.
CONF_THRESHOLD = 0.50

# ------------------------------------------------------------
# Região OCR usada para segmentação de notícias
# ------------------------------------------------------------
# O OCR completo continua disponível em ocr_all.
# Para calcular temas/transições/blocos, usamos por defeito a zona inferior do ecrã,
# porque aí aparecem títulos, subtítulos e rodapés editoriais.
SEGMENTATION_REGION = "lower_third"   # opções: "lower_third" ou "full_screen"

# Para vídeos 1280x720, verificámos manualmente que y≈480 apanha títulos/lower-thirds
# como "NEGOCIAÇÕES / GREVE GERAL", além do rodapé clássico.
LOWER_THIRD_Y_MIN_PX = 480

# Versão normalizada equivalente, caso as bounding boxes estejam em [0, 1].
LOWER_THIRD_Y_MIN_REL = 0.67

# Usar y2 significa: entra qualquer OCR cuja caixa toque/intersecte a região inferior.
LOWER_THIRD_FILTER_METHOD = "y2_intersects"

# Opcional: excluir uma zona fixa na direita/esquerda, se houver muito ruído de relógio/logo.
# Mantemos desligado por defeito para não criar regras específicas demais para um canal.
USE_X_EXCLUSION = False
EXCLUDE_RIGHT_X_MIN_PX = 1100

# ------------------------------------------------------------
# Critérios de transição OCR
# ------------------------------------------------------------
# A ideia agora é separar:
# - sinais fortes: mudança lexical alta e mudança temática forte;
# - sinais auxiliares: volume, mudança relevante de zona, tema novo depois de Unknown persistente;
# - sinais de qualidade: confidence_drop não cria bloco, apenas alerta.
LEXICAL_CHANGE_QUANTILE = 0.85
VOLUME_CHANGE_QUANTILE = 0.85

# Score mínimo ponderado para marcar uma transição candidata.
# Como lexical/tema forte valem 2, threshold 3 significa:
# "tem de haver pelo menos um sinal forte + algum suporte auxiliar".
TRANSITION_SCORE_THRESHOLD = 3

# Unknown curto pode ser só falha temporária de OCR/classificação.
# Só conta como "new theme after unknown" se o período Unknown anterior durar pelo menos isto.
MIN_UNKNOWN_GAP_SECONDS = 15

# Para a versão consolidada, blocos muito curtos são micro-blocos.
# Eles podem ser úteis para localizar transições, mas não devem ser contados logo como notícias.
MIN_FINAL_BLOCK_SECONDS = 30

# Pesos explicáveis do score.
TRANSITION_WEIGHTS = {
    "lexical_change_high": 2,
    "strong_theme_change": 2,
    "volume_change_high": 1,
    # Depois de restringirmos a segmentação ao lower-third, a zona deixa de ser decisiva.
    # Mantemos major_zone_change como coluna de debug, mas com peso 0.
    "major_zone_change": 0,
    "new_theme_after_unknown_persistent": 1,
    # quality_warning/confidence_drop fica fora do score.
}

# Grupos temáticos relacionados.
# Mudanças dentro do mesmo grupo contam como mudança fraca, não como mudança temática forte.
RELATED_THEME_GROUPS = {
    "politics": {"Eleições/Campanha", "Governo/Partidos", "Sondagens"},
    "public_services": {"Saúde", "Educação", "Habitação"},
    "economy_work": {"Economia", "Greves/Trabalho"},
    "security_justice": {"Justiça/Segurança", "Incêndios/Proteção Civil"},
    "international": {"Internacional"},
    "sports": {"Desporto"},
    "culture": {"Cultura"},
    "environment": {"Ambiente"},
    "transport": {"Transportes"},
    "weather": {"Meteorologia"},
}

# Dimensões do frame.
# Deixa None para inferir a partir das bounding boxes.
FRAME_WIDTH = None
FRAME_HEIGHT = None

print("BASE_DIR:", BASE_DIR.resolve())
print("FEATURES_DIR exists:", FEATURES_DIR.exists())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())
print("WINDOW_SIZE_SECONDS:", WINDOW_SIZE_SECONDS)
print("TRANSITION_SCORE_THRESHOLD:", TRANSITION_SCORE_THRESHOLD)
print("MIN_UNKNOWN_GAP_SECONDS:", MIN_UNKNOWN_GAP_SECONDS)
print("MIN_FINAL_BLOCK_SECONDS:", MIN_FINAL_BLOCK_SECONDS)
print("SEGMENTATION_REGION:", SEGMENTATION_REGION)
print("LOWER_THIRD_Y_MIN_PX:", LOWER_THIRD_Y_MIN_PX)


## 1. Funções auxiliares de carregamento OCR

Esta secção tenta ser robusta a diferentes formatos do pickle OCR.

Casos suportados:

- dataframe com uma coluna `OCR` contendo uma lista de deteções por frame;
- cada deteção como dicionário com `text`, `confidence` e `locations`/`bbox`;
- `OCR` como dicionário com listas paralelas de textos, confidences e locations;
- nomes alternativos de colunas, sempre que possível.


In [ ]:
def parse_ocr_filename(path):
    path = Path(path)
    stem = path.stem.replace("_ocr", "")
    parts = stem.split("_")

    # Esperado: Telejornal_RTP_Nov_20_ocr.pkl
    channel = parts[1] if len(parts) >= 2 else None
    month = parts[2] if len(parts) >= 3 else None
    day = parts[3] if len(parts) >= 4 else None

    return {
        "file": path.name,
        "channel": channel,
        "month": month,
        "day": day,
        "date_label": f"{month}_{day}" if month and day else None,
    }


def safe_float(x):
    try:
        if x is None:
            return np.nan
        if isinstance(x, str) and x.strip() == "":
            return np.nan
        return float(x)
    except Exception:
        return np.nan


def extract_frame_number(frame_value, fallback_index):
    # Extrai número do frame mesmo que venha como path/string.
    if isinstance(frame_value, (str, Path)):
        nums = re.findall(r"\d+", Path(str(frame_value)).stem)
        if nums:
            return int(nums[-1])
        return int(fallback_index)

    try:
        return int(float(frame_value))
    except Exception:
        return int(fallback_index)


def is_sequence(x):
    return isinstance(x, (list, tuple, np.ndarray, pd.Series)) and not isinstance(x, (str, bytes))


def first_existing_key(d, keys):
    if not isinstance(d, dict):
        return None
    for key in keys:
        if key in d:
            return key
    return None


def extract_text(det):
    if isinstance(det, dict):
        key = first_existing_key(det, ["text", "Text", "ocr_text", "OCR_text", "word", "value", "label"])
        if key is not None:
            return str(det[key])
    if isinstance(det, str):
        return det
    return None


def extract_conf(det):
    if isinstance(det, dict):
        key = first_existing_key(det, ["confidence", "conf", "score", "prob", "probability"])
        if key is not None:
            return safe_float(det[key])
    return np.nan


def normalize_bbox_vector(bbox):
    # Tenta converter uma bbox para [x1, y1, x2, y2].
    if bbox is None:
        return [np.nan, np.nan, np.nan, np.nan]

    # dict com coordenadas explícitas
    if isinstance(bbox, dict):
        key_variants = [
            ("x1", "y1", "x2", "y2"),
            ("left", "top", "right", "bottom"),
            ("xmin", "ymin", "xmax", "ymax"),
        ]
        for keys in key_variants:
            if all(k in bbox for k in keys):
                return [safe_float(bbox[k]) for k in keys]

    # numpy/lista/tuplo
    if is_sequence(bbox):
        arr = list(bbox)

        # Caso venha como [[x1,y1], [x2,y2], ...]
        if len(arr) > 0 and is_sequence(arr[0]):
            points = np.array(arr, dtype=float)
            if points.ndim == 2 and points.shape[1] >= 2:
                xs = points[:, 0]
                ys = points[:, 1]
                return [float(np.nanmin(xs)), float(np.nanmin(ys)), float(np.nanmax(xs)), float(np.nanmax(ys))]

        # Caso venha como [x1,y1,x2,y2]
        if len(arr) >= 4:
            return [safe_float(arr[0]), safe_float(arr[1]), safe_float(arr[2]), safe_float(arr[3])]

    return [np.nan, np.nan, np.nan, np.nan]


def extract_bbox(det):
    if isinstance(det, dict):
        key = first_existing_key(det, [
            "locations", "location", "bbox", "box", "bounding_box", "boundingBox", "points", "polygon"
        ])
        if key is not None:
            return normalize_bbox_vector(det[key])
    return [np.nan, np.nan, np.nan, np.nan]


def listify_value(x):
    if is_sequence(x):
        return list(x)
    return [x]


def expand_ocr_payload(payload):
    # Converte o conteúdo de uma célula OCR numa lista de deteções.
    # Suporta lista de dicionários, lista de strings ou dicionário com listas paralelas.
    if payload is None or (isinstance(payload, float) and np.isnan(payload)):
        return []

    # Caso 1: payload já é lista de deteções.
    if is_sequence(payload):
        return list(payload)

    # Caso 2: payload é dict com listas paralelas.
    if isinstance(payload, dict):
        text_key = first_existing_key(payload, ["text", "Text", "texts", "Texts", "words", "ocr_text"])
        conf_key = first_existing_key(payload, ["confidence", "conf", "confidences", "scores", "score", "probabilities"])
        loc_key = first_existing_key(payload, ["locations", "location", "bboxes", "boxes", "bbox", "bounding_boxes"])

        if text_key is not None and is_sequence(payload[text_key]):
            texts = listify_value(payload[text_key])
            confs = listify_value(payload[conf_key]) if conf_key is not None else [np.nan] * len(texts)
            locs = listify_value(payload[loc_key]) if loc_key is not None else [None] * len(texts)

            n = len(texts)
            detections = []
            for i in range(n):
                detections.append({
                    "text": texts[i],
                    "confidence": confs[i] if i < len(confs) else np.nan,
                    "locations": locs[i] if i < len(locs) else None,
                })
            return detections

        # Caso 3: payload é uma única deteção.
        return [payload]

    return []


def find_frame_column(df):
    candidates = ["Frame", "frame", "frame_id", "frame_number", "image", "image_path", "path"]
    for c in candidates:
        if c in df.columns:
            return c
    return df.columns[0]


def find_ocr_column(df):
    candidates = ["OCR", "ocr", "ocr_results", "detections", "texts", "text"]
    for c in candidates:
        if c in df.columns:
            return c
    return None


def normalize_ocr_df(df, file_name):
    # Transforma o OCR de um pickle numa tabela longa, uma linha por deteção OCR.
    meta = parse_ocr_filename(file_name)
    rows = []

    frame_col = find_frame_column(df)
    ocr_col = find_ocr_column(df)

    if ocr_col is None:
        raise ValueError(f"Não encontrei coluna OCR em {file_name}. Colunas: {list(df.columns)}")

    for idx, row in df.iterrows():
        frame_original = row[frame_col]
        frame_number = extract_frame_number(frame_original, idx)
        detections = expand_ocr_payload(row[ocr_col])

        for det in detections:
            text = extract_text(det)
            if text is None or len(str(text).strip()) == 0:
                continue

            x1, y1, x2, y2 = extract_bbox(det)
            conf = extract_conf(det)

            rows.append({
                "file": file_name,
                "channel": meta["channel"],
                "date_label": meta["date_label"],
                "frame": frame_number,
                "frame_original": str(frame_original),
                "text": str(text),
                "confidence": conf,
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
            })

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    out["second"] = out["frame"] / SAMPLE_RATE_FPS
    out["minute"] = (out["second"] // 60).astype(int)
    out["window_start_sec"] = (out["second"] // WINDOW_SIZE_SECONDS * WINDOW_SIZE_SECONDS).astype(int)
    out["window_end_sec"] = out["window_start_sec"] + WINDOW_SIZE_SECONDS
    out["minute_start_sec"] = (out["second"] // MINUTE_SIZE_SECONDS * MINUTE_SIZE_SECONDS).astype(int)
    return out


## 2. Carregar o telejornal escolhido

Esta secção carrega apenas **um** ficheiro OCR, porque a parte 2 é *single video analysis*.


In [ ]:
ocr_files = sorted(FEATURES_DIR.glob("*_ocr.pkl"))

print("N OCR files found:", len(ocr_files))
if ocr_files:
    print("Available OCR files:")
    for f in ocr_files[:50]:
        print(" -", f.name)

requested_path = FEATURES_DIR / TARGET_FILE

if requested_path.exists():
    target_path = requested_path
else:
    if len(ocr_files) == 0:
        raise FileNotFoundError(
            f"Não encontrei ficheiros *_ocr.pkl em {FEATURES_DIR.resolve()}. "
            "Confirma o caminho FEATURES_DIR."
        )
    print("\nWARNING: TARGET_FILE não encontrado:", TARGET_FILE)
    print("Vou usar o primeiro ficheiro OCR disponível. Altera TARGET_FILE para o ficheiro correto.")
    target_path = ocr_files[0]

print("\nUsing target OCR file:", target_path.name)

raw_ocr_df = pd.read_pickle(target_path)
print("Raw OCR dataframe shape:", raw_ocr_df.shape)
print("Raw OCR columns:", list(raw_ocr_df.columns))
display(raw_ocr_df.head(3))

ocr_long = normalize_ocr_df(raw_ocr_df, target_path.name)

print("\nNormalized OCR shape:", ocr_long.shape)
display(ocr_long.head(10))

if ocr_long.empty:
    raise ValueError("O OCR normalizado ficou vazio. Verifica a estrutura do pickle e as funções de extração.")


## 3. Limpeza de texto e tokenização

Aqui reutilizamos a lógica da parte 1: normalização de texto em português, stopwords e remoção de palavras estruturais/fixas.


In [ ]:
STOPWORDS_PT = {
    "de","a","o","e","que","do","da","em","um","uma","para","com","não","os","as","no","na","por",
    "se","ao","dos","das","mais","como","é","foi","são","ser","tem","também","ou","à","às","nos",
    "nas","sobre","entre","até","sem","já","lhe","ele","ela","eles","elas","sua","seu","suas","seus",
    "este","esta","estes","estas","isso","isto","há","vai","ter","mas","muito","muita","muitos","muitas",
    "porque","quando","onde","quem","qual","quais","todo","toda","todos","todas","num","numa","pelo","pela",
    "pelos","pelas","aos","ainda","só","era","foram","será","serem","nosso","nossa","seja","forma",
    "me", "te", "vos", "nos", "sido", "tinha", "tinham", "pode", "podem", "deve", "devem", "estar", "está",
    "estão", "faz", "fazer", "fez", "ano", "anos", "dia", "dias", "hoje", "ontem", "amanhã"
}

STRUCTURAL_WORDS = {
    "telejornal", "jornal", "nacional", "direto", "directo", "tvi", "rtp", "sic", "cnn", "cmtv",
    "notícias", "noticia", "noticias", "edição", "edicao", "especial", "última", "ultima", "hora", "minuto",
    "portugal", "portuguesa", "português", "portugues", "portugueses", "www", "pt", "hd", "direita", "esquerda",
    "legenda", "imagem", "arquivo", "fonte", "agência", "agencia"
}


def clean_text_pt(text):
    text = str(text).lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.replace("\\n", " ")
    text = re.sub(r"[^a-záàâãéèêíóôõúç0-9\s]", " ", text)
    text = re.sub(r"\b(b|br)\b", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize_pt(text, remove_stopwords=True, remove_structural=True, min_len=3):
    tokens = clean_text_pt(text).split()
    tokens = [t for t in tokens if len(t) >= min_len]

    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOPWORDS_PT]

    if remove_structural:
        tokens = [t for t in tokens if t not in STRUCTURAL_WORDS]

    return tokens


ocr_long["clean_text"] = ocr_long["text"].astype(str).apply(clean_text_pt)
ocr_long["tokens_content"] = ocr_long["clean_text"].apply(lambda x: tokenize_pt(x))
ocr_long["n_words_content"] = ocr_long["tokens_content"].apply(len)

# Remover linhas sem texto útil.
ocr_long = ocr_long[ocr_long["clean_text"].str.len() > 0].copy()

print("OCR rows after text cleaning:", len(ocr_long))
display(ocr_long[["frame", "second", "minute", "text", "clean_text", "confidence"]].head(10))


## 4. Dicionários de candidatos, partidos e temas

Estes dicionários vêm da lógica usada na parte 1, mas foram alargados para temas gerais de telejornal.

A parte 2 não deve ficar limitada a política, porque um telejornal pode ter saúde, internacional, economia, desporto, cultura, meteorologia, etc.


In [ ]:
# ------------------------------------------------------------
# Candidate aliases
# ------------------------------------------------------------
# Alguns aliases curtos podem gerar falsos positivos. Usar os resultados com cautela.

CANDIDATE_ALIASES = {
    "André Ventura": [
        "andré ventura", "andre ventura", "ventura", "líder do chega", "lider do chega"
    ],
    "Cotrim Figueiredo": [
        "cotrim", "cotrim de figueiredo", "cotrim figueiredo", "joão cotrim de figueiredo", "joao cotrim de figueiredo"
    ],
    "Luís Marques Mendes": [
        "marques mendes", "luís marques mendes", "luis marques mendes"
    ],
    "Henrique Gouveia e Melo": [
        "gouveia e melo", "gouveia melo", "henrique gouveia e melo", "almirante gouveia e melo"
    ],
    "António José Seguro": [
        "antónio josé seguro", "antonio jose seguro", "josé seguro", "jose seguro", "antónio seguro", "antonio seguro"
    ],
    "António Filipe": [
        "antónio filipe", "antonio filipe"
    ],
    "Catarina Martins": [
        "catarina martins", "ex coordenadora do bloco", "ex-coordenadora do bloco", "ex lider do bloco", "ex-lider do bloco"
    ],
    "Jorge Pinto": [
        "jorge pinto"
    ],
}

PARTY_ALIASES = {
    "CHEGA": ["chega", "partido chega"],
    "IL": ["il", "iniciativa liberal", "liberais"],
    "PSD/AD": ["psd", "ad", "aliança democrática", "alianca democratica", "partido social democrata", "sociais democratas"],
    "PS": ["ps", "partido socialista", "socialistas"],
    "PCP/CDU": ["pcp", "cdu", "partido comunista", "comunistas"],
    "BE": ["be", "bloco de esquerda"],
    "LIVRE": ["livre", "partido livre"],
    "CDS": ["cds", "cds pp", "cds-pp", "centro democrático social", "centro democratico social"],
}

THEME_ALIASES = {
    "Abertura/Manchetes": [
        "manchetes", "destaques", "sumário", "sumario",
        "em destaque", "abertura", "boa noite"
    ],

    "Eleições/Campanha": [
        "presidenciais", "presidencial",
        "eleições", "eleicoes", "eleição", "eleicao",
        "eleições presidenciais", "eleicoes presidenciais",
        "candidato", "candidata", "candidatos", "candidaturas",
        "campanha", "campanha eleitoral",
        "debate eleitoral", "debates eleitorais",
        "voto", "votos", "eleitores", "urna", "urnas"
    ],

    "Sondagens": [
        "sondagem", "sondagens",
        "barómetro", "barometro",
        "intenção de voto", "intencao de voto",
        "intenções de voto", "intencoes de voto",
        "estimativa eleitoral", "projeção eleitoral", "projecao eleitoral",
        "pontos percentuais"
    ],

    "Governo/Partidos": [
        "governo", "executivo",
        "primeiro ministro", "primeiro-ministro",
        "ministro", "ministra", "ministros",
        "parlamento", "assembleia da república", "assembleia da republica",
        "oposição", "oposicao",
        "líder parlamentar", "lider parlamentar",
        "grupo parlamentar", "maioria absoluta",
        "moção de censura", "mocao de censura",
        "partidos políticos", "partidos politicos"
    ],

    "Saúde": [
        "saúde", "saude",
        "sns", "serviço nacional de saúde", "servico nacional de saude",
        "hospital", "hospitais",
        "médico", "medico", "médicos", "medicos",
        "enfermeiro", "enfermeira", "enfermeiros", "enfermeiras",
        "urgência", "urgencia", "urgências", "urgencias",
        "doente", "doentes", "utente", "utentes",
        "vacina", "vacinas", "covid",
        "lista de espera", "listas de espera"
    ],

    "Economia/Proteção Social": [
        "economia", "económico", "economico",
        "inflação", "inflacao",
        "preços", "precos", "custo de vida",
        "impostos", "irs", "iva", "irc",
        "orçamento", "orcamento", "orçamento do estado", "orcamento do estado",
        "défice", "defice", "dívida pública", "divida publica",
        "salário", "salario", "salários", "salarios",
        "salário mínimo", "salario minimo",
        "pensões", "pensoes", "reformas",
        "segurança social", "seguranca social",
        "subsídio", "subsidio", "subsídios", "subsidios",
        "apoios sociais", "apoio social",
        "empresas", "juros", "taxas de juro",
        "banco de portugal", "bce", "bancos"
    ],

    "Habitação": [
        "habitação", "habitacao",
        "arrendamento", "arrendar",
        "renda", "rendas",
        "senhorio", "senhorios",
        "inquilino", "inquilinos",
        "crédito habitação", "credito habitacao",
        "empréstimo da casa", "emprestimo da casa",
        "preço das casas", "precos das casas",
        "mercado imobiliário", "mercado imobiliario",
        "imobiliário", "imobiliario"
    ],

    "Educação": [
        "educação", "educacao",
        "escola", "escolas",
        "professor", "professores",
        "aluno", "alunos",
        "ensino", "aulas",
        "creche", "creches",
        "universidade", "universidades",
        "estudante", "estudantes",
        "exames nacionais", "ano letivo", "ano lectivo"
    ],

    "Justiça/Segurança": [
        "justiça", "justica",
        "tribunal", "tribunais",
        "polícia", "policia",
        "psp", "gnr", "pj", "polícia judiciária", "policia judiciaria",
        "crime", "crimes", 
        "fraude", "fraudes",
        "sócrates", "socrates",
        "josé sócrates", "jose socrates",
        "homicídio", "homicidio",
        "agressão", "agressao",
        "corrupção", "corrupcao",
        "pgr", "ministério público", "ministerio publico",
        "detido", "detidos", "arguido", "arguidos",
        "prisão", "prisao",
        "caução", "caucao",
        "buscas", "operação policial", "operacao policial",

        # Acidentes e segurança pública
        "acidente", "acidentes",
        "colisão", "colisao",
        "despiste",
        "queda",
        "descarrilamento",
        "vítima", "vitima", "vítimas", "vitimas",
        "ferido", "feridos", "ferida", "feridas",
        "morto", "mortos", "morta", "mortas",
        "falha de segurança", "falha de seguranca",
        "falha técnica", "falha tecnica",
        "investigação", "investigacao",
        "inquérito", "inquerito"
    ],

    "Internacional": [
        "ucrânia", "ucrania",
        "rússia", "russia",
        "guerra na ucrânia", "guerra na ucrania",
        "israel", "gaza", "palestina", "hamas",
        "médio oriente", "medio oriente",
        "trump", "donald trump",
        "casa branca",
        "eua", "estados unidos",
        "bruxelas", "união europeia", "uniao europeia",
        "diplomacia", "diplomacia europeia",
        "diplomático", "diplomatico",
        "diplomática", "diplomatica",
        "relações diplomáticas", "relacoes diplomaticas",
        "nato", "onu",
        "frança", "franca",
        "espanha", "brasil", "china",
        "reino unido", "alemanha",
        "áfrica do sul", "africa do sul",
        "g20", "g7",

        # Política norte-americana
        "republicanos", "republicano",
        "partido republicano",
        "democratas", "democrata",
        "partido democrata",
        "congresso americano",
        "senado americano",
        "câmara dos representantes", "camara dos representantes"
    ],

    "Greves/Trabalho": [
        "greve", "greves",
        "sindicato", "sindicatos",
        "trabalhador", "trabalhadores",
        "protesto", "protestos",
        "manifestação", "manifestacao",
        "manifestantes",
        "contrato coletivo", "contrato colectivo",
        "concertação social", "concertacao social"
    ],

    "Transportes/Mobilidade": [
        "transportes",
        "metro", "metropolitano",
        "comboio", "comboios",
        "cp", "fertagus",
        "autocarro", "autocarros",
        "trânsito", "transito",
        "aeroporto",
        "tap", "tap air portugal",
        "avião", "aviao", "aviões", "avioes",
        "estrada", "autoestrada",
        "portagens",

        # Operadores e transportes urbanos
        "carris",
        "elétrico", "eletrico",
        "ascensor", "ascensores",
        "funicular",

        # Calçada/Elevador da Glória
        "calçada da glória", "calcada da gloria",
        "elevador da glória", "elevador da gloria"
    ],

    "Ambiente/Meteorologia/Proteção Civil": [
        "ambiente",
        "clima", "climático", "climatico",
        "alterações climáticas", "alteracoes climaticas",
        "seca", "chuva", "temporal",
        "inundações", "inundacoes", "cheias",
        "emissões", "emissoes", "poluição", "poluicao",
        "meteorologia", "previsão meteorológica", "previsao meteorologica",
        "temperatura", "vento", "frio", "calor", "neve",
        "incêndio", "incendio", "incêndios", "incendios",
        "bombeiros",
        "proteção civil", "protecao civil",
        "chamas", "evacuação", "evacuacao"
    ],

    "Desporto": [
        "futebol",
        "benfica", "sporting", "fc porto",
        "liga dos campeões", "liga dos campeoes",
        "liga portuguesa",
        "campeonato nacional",
        "seleção nacional", "selecao nacional",
        "treinador", "jogador", "jogadores",
        "golo", "golos",
        "estádio", "estadio"
    ],

    "Cultura": [
        "cultura",
        "cinema", "teatro",
        "música", "musica",
        "festival", "festivais",
        "livro", "livros",
        "exposição", "exposicao",
        "museu", "museus",
        "artista", "artistas",
        "concerto", "concertos"
    ],
}



def normalize_alias(alias):
    return clean_text_pt(alias)


def contains_alias(text, aliases):
    if not isinstance(text, str):
        return False

    text = clean_text_pt(text)

    for alias in aliases:
        alias = normalize_alias(alias)
        if not alias:
            continue
        pattern = r"(?<!\w)" + re.escape(alias) + r"(?!\w)"
        if re.search(pattern, text):
            return True
    return False


def matched_labels(text, alias_dict):
    return [label for label, aliases in alias_dict.items() if contains_alias(text, aliases)]


def matched_aliases(text, alias_dict):
    matches = []
    clean = clean_text_pt(text)
    for label, aliases in alias_dict.items():
        for alias in aliases:
            alias_norm = normalize_alias(alias)
            if alias_norm and re.search(r"(?<!\w)" + re.escape(alias_norm) + r"(?!\w)", clean):
                matches.append((label, alias))
    return matches

print("N candidates:", len(CANDIDATE_ALIASES))
print("N parties:", len(PARTY_ALIASES))
print("N themes:", len(THEME_ALIASES))


## 5. Confidence analysis

Antes de usar o OCR para segmentar o telejornal, avaliamos a qualidade das deteções.

Perguntas:

- Qual é a distribuição da confiança?
- Há minutos/janelas com OCR muito fraco?
- Quantas deteções ficam depois de aplicar o threshold?


In [ ]:
confidence_non_missing_ratio = ocr_long["confidence"].notna().mean()
print(f"Confidence non-missing ratio: {confidence_non_missing_ratio:.2%}")

if confidence_non_missing_ratio > 0:
    display(ocr_long["confidence"].describe())

    plt.figure(figsize=(8, 4))
    ocr_long["confidence"].dropna().hist(bins=30)
    plt.title("OCR confidence distribution")
    plt.xlabel("confidence")
    plt.ylabel("n_detections")
    plt.tight_layout()
    plt.show()
else:
    print("Não há valores de confidence disponíveis neste ficheiro.")

# Aplicar filtro apenas se houver confidence suficiente.
if confidence_non_missing_ratio >= 0.25:
    ocr_filtered = ocr_long[ocr_long["confidence"].fillna(0) >= CONF_THRESHOLD].copy()
    print(f"Applying confidence filter: confidence >= {CONF_THRESHOLD}")
else:
    ocr_filtered = ocr_long.copy()
    print("Confidence muito ausente; não vou filtrar por confidence.")

print("Rows before filter:", len(ocr_long))
print("Rows after filter:", len(ocr_filtered))
print("Kept ratio:", len(ocr_filtered) / max(len(ocr_long), 1))

confidence_by_minute = (
    ocr_long.groupby("minute")
    .agg(
        n_detections=("text", "size"),
        avg_confidence=("confidence", "mean"),
        median_confidence=("confidence", "median"),
        high_conf_ratio=("confidence", lambda s: float((s >= CONF_THRESHOLD).mean()) if s.notna().any() else np.nan),
        n_unique_texts=("clean_text", "nunique"),
    )
    .reset_index()
)

display(confidence_by_minute.head(15))

plt.figure(figsize=(12, 4))
plt.plot(confidence_by_minute["minute"], confidence_by_minute["n_detections"], marker="o")
plt.title("OCR volume by minute")
plt.xlabel("minute")
plt.ylabel("n_detections")
plt.tight_layout()
plt.show()

if confidence_non_missing_ratio > 0:
    plt.figure(figsize=(12, 4))
    plt.plot(confidence_by_minute["minute"], confidence_by_minute["avg_confidence"], marker="o")
    plt.title("Average OCR confidence by minute")
    plt.xlabel("minute")
    plt.ylabel("avg_confidence")
    plt.tight_layout()
    plt.show()

confidence_by_minute.to_csv(OUTPUT_DIR / "ocr_confidence_by_minute.csv", index=False)


## 6. Location analysis

Usamos as bounding boxes para transformar localização em zonas do ecrã.

Interpretação esperada:

- `bottom`: rodapés, legendas, identificação de pessoas;
- `middle/center`: títulos, gráficos, imagens de apoio;
- `top`: canal, relógio, manchetes ou elementos fixos.

Mudanças fortes de zona podem ser indícios de mudança de bloco/notícia.


In [ ]:
def infer_dimension(series, configured_value=None):
    if configured_value is not None:
        return float(configured_value)

    valid = pd.to_numeric(series, errors="coerce").dropna()
    if len(valid) == 0:
        return np.nan

    max_val = valid.quantile(0.99)

    # Se as coordenadas forem normalizadas.
    if max_val <= 1.5:
        return 1.0

    return float(max_val)


def add_bbox_features(df):
    df = df.copy()
    for c in ["x1", "y1", "x2", "y2"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["bbox_available"] = df[["x1", "y1", "x2", "y2"]].notna().all(axis=1)
    df["x_center"] = (df["x1"] + df["x2"]) / 2
    df["y_center"] = (df["y1"] + df["y2"]) / 2
    df["bbox_width"] = (df["x2"] - df["x1"]).abs()
    df["bbox_height"] = (df["y2"] - df["y1"]).abs()
    df["bbox_area"] = df["bbox_width"] * df["bbox_height"]

    inferred_width = infer_dimension(df["x2"], FRAME_WIDTH)
    inferred_height = infer_dimension(df["y2"], FRAME_HEIGHT)

    df.attrs["frame_width_inferred"] = inferred_width
    df.attrs["frame_height_inferred"] = inferred_height

    if not np.isnan(inferred_width) and inferred_width > 0:
        df["x_rel"] = df["x_center"] / inferred_width
    else:
        df["x_rel"] = np.nan

    if not np.isnan(inferred_height) and inferred_height > 0:
        df["y_rel"] = df["y_center"] / inferred_height
    else:
        df["y_rel"] = np.nan

    def vertical_zone(y):
        if pd.isna(y):
            return "unknown"
        if y < 1/3:
            return "top"
        if y < 2/3:
            return "middle"
        return "bottom"

    def horizontal_zone(x):
        if pd.isna(x):
            return "unknown"
        if x < 1/3:
            return "left"
        if x < 2/3:
            return "center"
        return "right"

    df["zone_vertical"] = df["y_rel"].apply(vertical_zone)
    df["zone_horizontal"] = df["x_rel"].apply(horizontal_zone)
    df["screen_zone"] = df["zone_vertical"] + "_" + df["zone_horizontal"]
    return df


ocr_filtered = add_bbox_features(ocr_filtered)

print("Inferred frame width:", ocr_filtered.attrs.get("frame_width_inferred"))
print("Inferred frame height:", ocr_filtered.attrs.get("frame_height_inferred"))
print("BBox available ratio:", f"{ocr_filtered['bbox_available'].mean():.2%}")

display(ocr_filtered[[
    "frame", "text", "confidence", "x1", "y1", "x2", "y2", "zone_vertical", "zone_horizontal", "screen_zone"
]].head(10))

zone_counts = (
    ocr_filtered.groupby(["zone_vertical", "zone_horizontal", "screen_zone"])
    .size()
    .reset_index(name="n_detections")
    .sort_values("n_detections", ascending=False)
)

display(zone_counts)

plt.figure(figsize=(8, 4))
zone_counts.head(12).plot(kind="bar", x="screen_zone", y="n_detections", legend=False, ax=plt.gca())
plt.title("OCR detections by screen zone")
plt.xlabel("screen_zone")
plt.ylabel("n_detections")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

zone_by_minute = (
    ocr_filtered.groupby(["minute", "zone_vertical"])
    .size()
    .reset_index(name="n_detections")
)

zone_by_minute_pivot = zone_by_minute.pivot(index="minute", columns="zone_vertical", values="n_detections").fillna(0)

display(zone_by_minute_pivot.head(15))

plt.figure(figsize=(12, 4))
for col in zone_by_minute_pivot.columns:
    plt.plot(zone_by_minute_pivot.index, zone_by_minute_pivot[col], marker="o", label=col)
plt.title("OCR vertical zones by minute")
plt.xlabel("minute")
plt.ylabel("n_detections")
plt.legend()
plt.tight_layout()
plt.show()

zone_counts.to_csv(OUTPUT_DIR / "ocr_zone_counts.csv", index=False)
zone_by_minute_pivot.to_csv(OUTPUT_DIR / "ocr_zone_by_minute.csv")


## 6.1 Região OCR usada para segmentação

A análise de qualidade e localização usa o OCR completo filtrado por confiança (`ocr_all`).

Para **temas, mudanças lexicais, transições e blocos/notícias**, usamos por defeito uma versão mais focada: `ocr_for_segmentation`.

Nesta versão, `ocr_for_segmentation` contém apenas OCR que intersecta a zona inferior do ecrã. A regra principal é:

```python
y2 >= 480
```

para vídeos 720p. Isto permite apanhar tanto o rodapé clássico como títulos um pouco mais altos, por exemplo `NEGOCIAÇÕES / GREVE GERAL`, reduzindo ruído vindo do resto do ecrã.


In [ ]:
# ------------------------------------------------------------
# OCR completo vs OCR usado para segmentação
# ------------------------------------------------------------
# ocr_all: OCR completo após filtro de confiança e features de localização.
# ocr_for_segmentation: OCR usado para calcular temas/transições/blocos.
# Por defeito, é apenas OCR que intersecta a zona inferior do ecrã.

ocr_all = ocr_filtered.copy()

inferred_height = ocr_all.attrs.get("frame_height_inferred", np.nan)
inferred_width = ocr_all.attrs.get("frame_width_inferred", np.nan)

# Determinar threshold vertical dependendo de coordenadas em pixels ou normalizadas.
coords_look_normalized = (
    pd.notna(inferred_height) and inferred_height <= 1.5
)

if coords_look_normalized:
    lower_third_y_min = LOWER_THIRD_Y_MIN_REL
    exclude_right_x_min = EXCLUDE_RIGHT_X_MIN_PX / 1280  # fallback aproximado se necessário
else:
    lower_third_y_min = LOWER_THIRD_Y_MIN_PX
    exclude_right_x_min = EXCLUDE_RIGHT_X_MIN_PX

if SEGMENTATION_REGION == "lower_third":
    if LOWER_THIRD_FILTER_METHOD == "y2_intersects":
        # Entra qualquer OCR cuja bounding box toque a zona inferior.
        ocr_for_segmentation = ocr_all[ocr_all["y2"] >= lower_third_y_min].copy()
    elif LOWER_THIRD_FILTER_METHOD == "y_center":
        # Alternativa mais restritiva: usa o centro da bounding box.
        ocr_for_segmentation = ocr_all[ocr_all["y_center"] >= lower_third_y_min].copy()
    else:
        raise ValueError("LOWER_THIRD_FILTER_METHOD deve ser 'y2_intersects' ou 'y_center'.")
elif SEGMENTATION_REGION == "full_screen":
    ocr_for_segmentation = ocr_all.copy()
else:
    raise ValueError("SEGMENTATION_REGION deve ser 'lower_third' ou 'full_screen'.")

if USE_X_EXCLUSION:
    # Opcional: remover texto fixo muito à direita, como relógio/logo, se estiver a causar ruído.
    ocr_for_segmentation = ocr_for_segmentation[
        ocr_for_segmentation["x_center"] < exclude_right_x_min
    ].copy()

ocr_for_segmentation["segmentation_region"] = SEGMENTATION_REGION

print("OCR completo após confidence/location:", len(ocr_all))
print("OCR usado para segmentação:", len(ocr_for_segmentation))
print("Kept ratio for segmentation:", len(ocr_for_segmentation) / max(len(ocr_all), 1))
print("Coordinates look normalized:", coords_look_normalized)
print("Segmentation y threshold used:", lower_third_y_min)
print("Segmentation filter method:", LOWER_THIRD_FILTER_METHOD)

display(ocr_for_segmentation[[
    "frame", "second", "text", "confidence", "x1", "y1", "x2", "y2",
    "screen_zone", "segmentation_region"
]].head(20))

# Comparação rápida por minuto: OCR completo vs OCR usado na segmentação.
segmentation_volume_compare = pd.DataFrame({
    "all_filtered": ocr_all.groupby("minute").size(),
    "segmentation_region": ocr_for_segmentation.groupby("minute").size(),
}).fillna(0).reset_index().rename(columns={"index": "minute"})

display(segmentation_volume_compare.head(20))

plt.figure(figsize=(12, 4))
plt.plot(segmentation_volume_compare["minute"], segmentation_volume_compare["all_filtered"], marker="o", label="all OCR")
plt.plot(segmentation_volume_compare["minute"], segmentation_volume_compare["segmentation_region"], marker="o", label="segmentation OCR")
plt.title("OCR volume: full screen vs segmentation region")
plt.xlabel("minute")
plt.ylabel("n_detections")
plt.legend()
plt.tight_layout()
plt.show()

ocr_for_segmentation.to_csv(OUTPUT_DIR / "ocr_segmentation_region_used.csv", index=False)
segmentation_volume_compare.to_csv(OUTPUT_DIR / "ocr_segmentation_region_volume_compare.csv", index=False)


## 7. Deduplicação de texto OCR na região de segmentação

A partir daqui, as análises de tema/transição/blocos usam `ocr_for_segmentation`, por defeito o OCR da zona inferior do ecrã.

Isto evita que texto de gráficos, logos, fundos ou outros elementos fora do rodapé influencie diretamente a divisão das notícias.


In [ ]:
# Deduplicação simples: dentro da mesma janela e zona, contar cada texto limpo apenas uma vez.
# Importante: a partir daqui usamos ocr_for_segmentation, não o OCR completo.
# Isto significa que temas/transições/blocos são calculados sobretudo a partir do lower-third/rodapé.

ocr_dedup = (
    ocr_for_segmentation
    .sort_values(["window_start_sec", "frame", "screen_zone", "clean_text"])
    .drop_duplicates(subset=["window_start_sec", "screen_zone", "clean_text"])
    .copy()
)

print("Rows after confidence/location processing (full OCR):", len(ocr_all))
print("Rows used for segmentation before dedup:", len(ocr_for_segmentation))
print("Rows used for segmentation after dedup:", len(ocr_dedup))
print("Segmentation kept ratio vs full OCR:", len(ocr_for_segmentation) / max(len(ocr_all), 1))
print("Dedup kept ratio inside segmentation region:", len(ocr_dedup) / max(len(ocr_for_segmentation), 1))

# Comparar volume antes/depois.
volume_compare = pd.DataFrame({
    "full_filtered": ocr_all.groupby("minute").size(),
    "segmentation_region": ocr_for_segmentation.groupby("minute").size(),
    "dedup_segmentation": ocr_dedup.groupby("minute").size(),
}).fillna(0).reset_index().rename(columns={"index": "minute"})

display(volume_compare.head(15))

plt.figure(figsize=(12, 4))
plt.plot(volume_compare["minute"], volume_compare["full_filtered"], marker="o", label="full OCR")
plt.plot(volume_compare["minute"], volume_compare["segmentation_region"], marker="o", label="segmentation region")
plt.plot(volume_compare["minute"], volume_compare["dedup_segmentation"], marker="o", label="dedup segmentation")
plt.title("OCR volume before vs after lower-third selection and deduplication")
plt.xlabel("minute")
plt.ylabel("n_detections")
plt.legend()
plt.tight_layout()
plt.show()

ocr_dedup.to_csv(OUTPUT_DIR / "ocr_single_newscast_lower_third_dedup.csv", index=False)
# Nome antigo mantido para compatibilidade com versões anteriores.
ocr_dedup.to_csv(OUTPUT_DIR / "ocr_single_newscast_long_dedup.csv", index=False)


## 8. Timeline temática baseada no OCR do lower-third

Esta timeline é calculada a partir de `ocr_dedup`, ou seja, OCR deduplicado da região usada para segmentação.

Nesta versão, o tema dominante de cada janela de 3 segundos é inferido sobretudo a partir do rodapé/títulos/subtítulos, não do ecrã completo.


In [ ]:
def seconds_to_hhmmss(seconds):
    seconds = int(seconds)
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    if h > 0:
        return f"{h:02d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"


def labels_in_texts(texts, alias_dict):
    present = []
    text_concat = " ".join([str(t) for t in texts])
    for label, aliases in alias_dict.items():
        if contains_alias(text_concat, aliases):
            present.append(label)
    return present


def count_label_hits(group, alias_dict):
    counts = {}
    for label, aliases in alias_dict.items():
        counts[label] = int(group["clean_text"].apply(lambda t: contains_alias(t, aliases)).sum())
    return counts


def top_example_texts(group, max_items=5, max_chars=350):
    texts = []
    for t in group["clean_text"].dropna().astype(str):
        if len(t) < 3:
            continue
        if t not in texts:
            texts.append(t)
        if len(texts) >= max_items:
            break
    out = " | ".join(texts)
    return out[:max_chars]


print('Fonte usada para timeline temática:', SEGMENTATION_REGION)
print('Deteções deduplicadas usadas:', len(ocr_dedup))

window_rows = []
theme_long_rows = []

for window_start, group in ocr_dedup.groupby("window_start_sec"):
    group = group.sort_values("frame")
    theme_counts = count_label_hits(group, THEME_ALIASES)
    candidate_counts = count_label_hits(group, CANDIDATE_ALIASES)
    party_counts = count_label_hits(group, PARTY_ALIASES)

    theme_hits_total = sum(theme_counts.values())
    if theme_hits_total > 0:
        dominant_theme = max(theme_counts, key=theme_counts.get)
        dominant_theme_score = theme_counts[dominant_theme]
    else:
        dominant_theme = "Other/Unknown"
        dominant_theme_score = 0

    zone_mode = group["screen_zone"].mode().iloc[0] if "screen_zone" in group and not group["screen_zone"].mode().empty else "unknown"
    vertical_mode = group["zone_vertical"].mode().iloc[0] if "zone_vertical" in group and not group["zone_vertical"].mode().empty else "unknown"

    window_rows.append({
        "window_start_sec": int(window_start),
        "window_end_sec": int(window_start + WINDOW_SIZE_SECONDS),
        "time_start": seconds_to_hhmmss(window_start),
        "time_end": seconds_to_hhmmss(window_start + WINDOW_SIZE_SECONDS),
        "minute": int(window_start // 60),
        "n_detections": int(len(group)),
        "n_unique_texts": int(group["clean_text"].nunique()),
        "n_words": int(group["n_words_content"].sum()),
        "avg_confidence": group["confidence"].mean(),
        "dominant_theme": dominant_theme,
        "dominant_theme_score": int(dominant_theme_score),
        "theme_hits_total": int(theme_hits_total),
        "themes_present": ", ".join([k for k, v in theme_counts.items() if v > 0]),
        "candidates_present": ", ".join([k for k, v in candidate_counts.items() if v > 0]),
        "parties_present": ", ".join([k for k, v in party_counts.items() if v > 0]),
        "dominant_screen_zone": zone_mode,
        "dominant_vertical_zone": vertical_mode,
        "example_text": top_example_texts(group),
        "text_concat": " ".join(group["clean_text"].astype(str).tolist()),
    })

    for theme, hits in theme_counts.items():
        theme_long_rows.append({
            "window_start_sec": int(window_start),
            "time_start": seconds_to_hhmmss(window_start),
            "theme": theme,
            "hits": int(hits),
        })

window_timeline = pd.DataFrame(window_rows).sort_values("window_start_sec").reset_index(drop=True)
theme_timeline_long = pd.DataFrame(theme_long_rows)

display(window_timeline[[
    "time_start", "time_end", "n_detections", "avg_confidence", "dominant_theme", "dominant_theme_score",
    "themes_present", "candidates_present", "dominant_screen_zone", "example_text"
]].head(20))

# Guardar.
window_timeline.to_csv(OUTPUT_DIR / "ocr_window_topic_timeline.csv", index=False)
theme_timeline_long.to_csv(OUTPUT_DIR / "ocr_theme_timeline_long.csv", index=False)


In [ ]:
# Plot: top temas ao longo do tempo.
if not theme_timeline_long.empty:
    top_themes = (
        theme_timeline_long.groupby("theme")["hits"]
        .sum()
        .sort_values(ascending=False)
        .head(8)
        .index
        .tolist()
    )

    theme_wide = (
        theme_timeline_long[theme_timeline_long["theme"].isin(top_themes)]
        .pivot_table(index="window_start_sec", columns="theme", values="hits", aggfunc="sum")
        .fillna(0)
        .sort_index()
    )

    display(theme_wide.head(15))

    plt.figure(figsize=(14, 5))
    for theme in theme_wide.columns:
        plt.plot(theme_wide.index / 60, theme_wide[theme], marker="o", label=theme)
    plt.title("Lower-third OCR topic mentions over time")
    plt.xlabel("minute")
    plt.ylabel("theme hits")
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.show()
else:
    print("theme_timeline_long está vazio.")


## 9. Mudança lexical entre janelas

A análise por dicionário é explicável, mas pode falhar quando aparece um tema não coberto.

Por isso, medimos também a mudança lexical entre janelas consecutivas com TF-IDF + cosine similarity.

Interpretação:

- `lexical_similarity` alto → texto semelhante à janela anterior;
- `lexical_change` alto → possível mudança de notícia/bloco.


In [ ]:
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    SKLEARN_AVAILABLE = True
except Exception as e:
    SKLEARN_AVAILABLE = False
    print("sklearn não disponível. A mudança lexical será ignorada.")
    print(e)

window_timeline["lexical_similarity_prev"] = np.nan
window_timeline["lexical_change"] = np.nan

if SKLEARN_AVAILABLE and len(window_timeline) >= 2:
    docs = window_timeline["text_concat"].fillna("").astype(str).tolist()

    # min_df=1 porque há apenas um telejornal; ngram_range ajuda a capturar expressões curtas.
    vectorizer = TfidfVectorizer(
        tokenizer=lambda x: tokenize_pt(x),
        token_pattern=None,
        min_df=1,
        ngram_range=(1, 2)
    )

    try:
        X = vectorizer.fit_transform(docs)
        sims = [np.nan]
        for i in range(1, X.shape[0]):
            sim = cosine_similarity(X[i], X[i-1])[0, 0]
            sims.append(float(sim))

        window_timeline["lexical_similarity_prev"] = sims
        window_timeline["lexical_change"] = 1 - window_timeline["lexical_similarity_prev"]
    except ValueError as e:
        print("Não foi possível calcular TF-IDF:", e)

lexical_cols = ["time_start", "dominant_theme", "lexical_similarity_prev", "lexical_change", "example_text"]
display(window_timeline[lexical_cols].head(20))

if window_timeline["lexical_change"].notna().any():
    plt.figure(figsize=(20, 4))
    plt.plot(window_timeline["window_start_sec"] / 60, window_timeline["lexical_change"], marker="o")
    plt.title("Lexical change between consecutive OCR windows")
    plt.xlabel("minute")
    plt.ylabel("lexical_change = 1 - cosine_similarity")
    plt.tight_layout()
    plt.show()

window_timeline.to_csv(OUTPUT_DIR / "ocr_window_topic_timeline_with_lexical_change.csv", index=False)


## 10. OCR-based transition score — versão ponderada com lower-third

O score de transição é calculado com base no OCR da região de segmentação (`ocr_for_segmentation` → `ocr_dedup`).

Nesta versão:

- `lexical_change_high` e `strong_theme_change` são sinais fortes;
- `volume_change_high` é sinal auxiliar;
- `new_theme_after_unknown_persistent` só conta se o período `Unknown` for suficientemente longo;
- `major_zone_change` fica como debug, mas tem peso 0 porque já estamos focados na zona inferior;
- `confidence_drop` é apenas aviso de qualidade, não cria blocos.


In [ ]:
def bool_to_int(x):
    return int(bool(x))


def theme_group(theme):
    """Mapeia um tema para o grupo temático definido na configuração."""
    if pd.isna(theme) or theme == "Other/Unknown":
        return "unknown"
    for group_name, themes in RELATED_THEME_GROUPS.items():
        if theme in themes:
            return group_name
    return str(theme)


def split_vertical_zone(screen_zone):
    """Extrai a zona vertical de uma string como bottom_right."""
    if pd.isna(screen_zone):
        return "unknown"
    screen_zone = str(screen_zone)
    if "_" not in screen_zone:
        return screen_zone
    return screen_zone.split("_")[0]


# Sinais básicos.
window_timeline = window_timeline.sort_values("window_start_sec").reset_index(drop=True)

window_timeline["prev_theme"] = window_timeline["dominant_theme"].shift(1)
window_timeline["prev_theme_group"] = window_timeline["prev_theme"].apply(theme_group)
window_timeline["theme_group"] = window_timeline["dominant_theme"].apply(theme_group)

window_timeline["prev_zone"] = window_timeline["dominant_screen_zone"].shift(1)
window_timeline["prev_vertical_zone"] = window_timeline["prev_zone"].apply(split_vertical_zone)
window_timeline["vertical_zone"] = window_timeline["dominant_screen_zone"].apply(split_vertical_zone)

window_timeline["prev_avg_confidence"] = window_timeline["avg_confidence"].shift(1)
window_timeline["prev_n_detections"] = window_timeline["n_detections"].shift(1)

# ------------------------------------------------------------
# 1) Mudança temática forte
# ------------------------------------------------------------
# Só conta como forte quando:
# - o tema anterior e atual são conhecidos;
# - são diferentes;
# - pertencem a grupos temáticos diferentes.
# Assim, Governo/Partidos <-> Eleições/Campanha não parte a notícia sozinho.
window_timeline["theme_changed"] = (
    (window_timeline["dominant_theme"] != window_timeline["prev_theme"]) &
    window_timeline["prev_theme"].notna() &
    (window_timeline["dominant_theme"] != "Other/Unknown")
)

window_timeline["strong_theme_change"] = (
    (window_timeline["dominant_theme"] != window_timeline["prev_theme"]) &
    window_timeline["prev_theme"].notna() &
    (window_timeline["dominant_theme"] != "Other/Unknown") &
    (window_timeline["prev_theme"] != "Other/Unknown") &
    (window_timeline["theme_group"] != window_timeline["prev_theme_group"])
)

# ------------------------------------------------------------
# 2) Mudança de zona
# ------------------------------------------------------------
# minor_zone_change: qualquer alteração, por exemplo bottom_center -> bottom_right.
# major_zone_change: mudança vertical relevante, por exemplo bottom -> middle.
window_timeline["minor_zone_change"] = (
    (window_timeline["dominant_screen_zone"] != window_timeline["prev_zone"]) &
    window_timeline["prev_zone"].notna() &
    (window_timeline["dominant_screen_zone"] != "unknown_unknown")
)

window_timeline["major_zone_change"] = (
    (window_timeline["vertical_zone"] != window_timeline["prev_vertical_zone"]) &
    window_timeline["prev_vertical_zone"].notna() &
    (~window_timeline["vertical_zone"].isin(["unknown", "unknown_unknown"])) &
    (~window_timeline["prev_vertical_zone"].isin(["unknown", "unknown_unknown"]))
)

# Para compatibilidade com código/prints anteriores.
window_timeline["zone_changed"] = window_timeline["major_zone_change"]

# ------------------------------------------------------------
# 3) Mudança de volume OCR
# ------------------------------------------------------------
window_timeline["volume_abs_change"] = (
    window_timeline["n_detections"] - window_timeline["prev_n_detections"]
).abs()

if window_timeline["lexical_change"].notna().any():
    lexical_thr = window_timeline["lexical_change"].dropna().quantile(LEXICAL_CHANGE_QUANTILE)
else:
    lexical_thr = np.inf

volume_thr = (
    window_timeline["volume_abs_change"].dropna().quantile(VOLUME_CHANGE_QUANTILE)
    if window_timeline["volume_abs_change"].notna().any()
    else np.inf
)

window_timeline["lexical_change_high"] = window_timeline["lexical_change"] >= lexical_thr
window_timeline["volume_change_high"] = window_timeline["volume_abs_change"] >= volume_thr

# ------------------------------------------------------------
# 4) Confidence drop
# ------------------------------------------------------------
# Isto é um aviso de qualidade, não um sinal de nova notícia.
window_timeline["confidence_drop"] = (
    window_timeline["prev_avg_confidence"].notna() &
    window_timeline["avg_confidence"].notna() &
    ((window_timeline["prev_avg_confidence"] - window_timeline["avg_confidence"]) >= 0.15)
)
window_timeline["quality_warning"] = window_timeline["confidence_drop"]

# ------------------------------------------------------------
# 5) Tema conhecido depois de Unknown persistente
# ------------------------------------------------------------
# Unknown curto pode ser só ruído/instabilidade do OCR.
unknown_run_seconds_before = []
unknown_run = 0

for _, row in window_timeline.iterrows():
    unknown_run_seconds_before.append(unknown_run)

    duration = int(row["window_end_sec"] - row["window_start_sec"])
    if row["dominant_theme"] == "Other/Unknown":
        unknown_run += duration
    else:
        unknown_run = 0

window_timeline["unknown_run_seconds_before"] = unknown_run_seconds_before

window_timeline["new_theme_after_unknown"] = (
    (window_timeline["prev_theme"] == "Other/Unknown") &
    (window_timeline["dominant_theme"] != "Other/Unknown")
)

window_timeline["new_theme_after_unknown_persistent"] = (
    window_timeline["new_theme_after_unknown"] &
    (window_timeline["unknown_run_seconds_before"] >= MIN_UNKNOWN_GAP_SECONDS)
)

# ------------------------------------------------------------
# Score ponderado
# ------------------------------------------------------------
# Nota: os sinais foram calculados a partir do OCR da região de segmentação.
# Como usamos sobretudo lower-third, major_zone_change tem peso 0 por defeito.

for col in TRANSITION_WEIGHTS:
    if col not in window_timeline.columns:
        window_timeline[col] = False

window_timeline["transition_score"] = 0
for col, weight in TRANSITION_WEIGHTS.items():
    window_timeline["transition_score"] += window_timeline[col].fillna(False).astype(int) * weight

# Tem de existir pelo menos um sinal forte.
window_timeline["has_strong_signal"] = (
    window_timeline["lexical_change_high"].fillna(False) |
    window_timeline["strong_theme_change"].fillna(False)
)

window_timeline["is_transition_candidate"] = (
    (window_timeline["transition_score"] >= TRANSITION_SCORE_THRESHOLD) &
    window_timeline["has_strong_signal"]
)

# A primeira linha nunca é transição candidata; o início do vídeo é tratado à parte.
window_timeline.loc[0, "transition_score"] = 0
window_timeline.loc[0, "is_transition_candidate"] = False


def transition_reasons(row):
    reasons = []

    if row.get("lexical_change_high", False):
        reasons.append(f"high lexical change (+{TRANSITION_WEIGHTS['lexical_change_high']})")

    if row.get("strong_theme_change", False):
        reasons.append(f"strong theme change (+{TRANSITION_WEIGHTS['strong_theme_change']})")
    elif row.get("theme_changed", False):
        reasons.append("weak/related theme change (+0)")

    if row.get("volume_change_high", False):
        reasons.append(f"OCR volume changed (+{TRANSITION_WEIGHTS['volume_change_high']})")

    if row.get("major_zone_change", False):
        reasons.append(f"major screen zone changed (+{TRANSITION_WEIGHTS.get('major_zone_change', 0)}, debug)")
    elif row.get("minor_zone_change", False):
        reasons.append("minor screen zone changed (+0, debug)")

    if row.get("new_theme_after_unknown_persistent", False):
        reasons.append(f"new theme after persistent unknown (+{TRANSITION_WEIGHTS['new_theme_after_unknown_persistent']})")
    elif row.get("new_theme_after_unknown", False):
        reasons.append("new theme after short unknown (+0)")

    if row.get("confidence_drop", False):
        reasons.append("confidence drop / quality warning (+0)")

    return "; ".join(reasons)


window_timeline["transition_reasons"] = window_timeline.apply(transition_reasons, axis=1)

transition_candidates = (
    window_timeline[window_timeline["is_transition_candidate"]]
    .copy()
    .sort_values(["transition_score", "window_start_sec"], ascending=[False, True])
)

print("Segmentation source:", SEGMENTATION_REGION)
print("Lexical change threshold:", lexical_thr)
print("Volume change threshold:", volume_thr)
print("Transition score threshold:", TRANSITION_SCORE_THRESHOLD)
print("Weights:", TRANSITION_WEIGHTS)
print("N transition candidates:", len(transition_candidates))

debug_cols = [
    "time_start", "time_end",
    "transition_score", "has_strong_signal", "is_transition_candidate",
    "transition_reasons",
    "prev_theme", "dominant_theme", "prev_theme_group", "theme_group",
    "prev_zone", "dominant_screen_zone", "minor_zone_change", "major_zone_change",
    "lexical_change", "volume_abs_change", "avg_confidence", "quality_warning",
    "unknown_run_seconds_before", "example_text",
]

display(transition_candidates[debug_cols].head(30))

plt.figure(figsize=(12, 4))
plt.plot(window_timeline["window_start_sec"] / 60, window_timeline["transition_score"], marker="o")
plt.axhline(TRANSITION_SCORE_THRESHOLD, linestyle="--")
plt.title("Weighted OCR-based transition score over time")
plt.xlabel("minute")
plt.ylabel("weighted transition_score")
plt.tight_layout()
plt.show()

window_timeline.to_csv(OUTPUT_DIR / "ocr_window_timeline_with_transition_score.csv", index=False)
transition_candidates.to_csv(OUTPUT_DIR / "ocr_transition_candidates.csv", index=False)


## 11. Micro-blocos OCR e blocos/notícias consolidados

Os micro-blocos são criados a partir dos candidatos de transição calculados com OCR do lower-third.

Como as janelas têm 3 segundos, estes micro-blocos são úteis para localizar mudanças finas, mas não devem ser automaticamente interpretados como notícias finais.

A tabela `consolidated_blocks` junta micro-blocos consecutivos quando há continuidade temática, entidades semelhantes ou compatibilidade temporal/localização, produzindo uma estimativa mais adequada para contar notícias e duração.


In [ ]:
# Ordenar cronologicamente e usar apenas as transições candidatas para iniciar micro-blocos.
blocks_df = window_timeline.sort_values("window_start_sec").copy()
blocks_df["is_transition_start"] = blocks_df["is_transition_candidate"]
blocks_df.loc[blocks_df.index[0], "is_transition_start"] = True
blocks_df["block_id"] = blocks_df["is_transition_start"].cumsum()


def mode_or_unknown(series):
    s = series.dropna()
    if len(s) == 0:
        return "unknown"
    return s.mode().iloc[0]


"""" ANtes tinha um problema em q numa janela q começava c mmudanca de tema e fosse segyuida por unkwons e no meio houveesse qql tema esse tema ia domingar a janbela mesmo c poucos hits. 
Agora o weighted_dominant_theme só aceita um tema se ele tiver persistência mínima, evitando que um tema com poucos hits e seguido de unknowns domine a janela.


def weighted_dominant_theme(group):
    # Escolhe o tema com maior soma de dominant_theme_score dentro do bloco.
    temp = group[group["dominant_theme"] != "Other/Unknown"].copy()
    if temp.empty:
        return "Other/Unknown"
    scores = temp.groupby("dominant_theme")["dominant_theme_score"].sum().sort_values(ascending=False)
    return scores.index[0]
"""

def weighted_dominant_theme(group, min_windows=3, min_ratio=0.10):
    """
    Escolhe o tema dominante do bloco, mas só aceita um tema se ele tiver persistência mínima.

    Isto evita que uma única janela de 3 segundos com 'Desporto' classifique
    um bloco inteiro de vários minutos como Desporto.
    """

    total_windows = len(group)

    # Ignorar Other/Unknown para procurar temas conhecidos,
    # mas não deixar um tema conhecido ganhar se aparecer só uma vez.
    temp = group[group["dominant_theme"] != "Other/Unknown"].copy()

    if temp.empty:
        return "Other/Unknown"

    stats = (
        temp.groupby("dominant_theme")
        .agg(
            score=("dominant_theme_score", "sum"),
            n_windows=("dominant_theme", "size")
        )
    )

    stats["coverage_ratio"] = stats["n_windows"] / max(total_windows, 1)

    stats = stats.sort_values(
        ["score", "n_windows", "coverage_ratio"],
        ascending=False
    )

    best_theme = stats.index[0]
    best_n_windows = stats.loc[best_theme, "n_windows"]
    best_coverage = stats.loc[best_theme, "coverage_ratio"]

    # Regra principal:
    # o tema tem de aparecer em pelo menos 3 janelas
    # e em pelo menos 10% do bloco.
    if best_n_windows < min_windows or best_coverage < min_ratio:
        return "Other/Unknown"

    return best_theme


    

def concat_unique_non_empty(values, max_items=8):
    out = []
    for v in values:
        if pd.isna(v) or str(v).strip() == "":
            continue
        for item in str(v).split(","):
            item = item.strip()
            if item and item not in out:
                out.append(item)
            if len(out) >= max_items:
                break
        if len(out) >= max_items:
            break
    return ", ".join(out)


def split_items(value):
    if pd.isna(value) or str(value).strip() == "":
        return set()
    return {item.strip() for item in str(value).split(",") if item.strip()}


def vertical_from_screen_zone(value):
    if pd.isna(value):
        return "unknown"
    value = str(value)
    return value.split("_")[0] if "_" in value else value


def themes_are_similar(row_a, row_b):
    """Critério conservador para decidir se dois blocos consecutivos parecem o mesmo assunto."""
    theme_a = row_a.get("dominant_theme", "Other/Unknown")
    theme_b = row_b.get("dominant_theme", "Other/Unknown")

    if theme_a != "Other/Unknown" and theme_a == theme_b:
        return True

    themes_a = split_items(row_a.get("themes_present", ""))
    themes_b = split_items(row_b.get("themes_present", ""))

    if len(themes_a & themes_b) > 0:
        return True

    # Temas políticos relacionados podem oscilar em janelas pequenas sem ser uma nova notícia.
    if theme_a != "Other/Unknown" and theme_b != "Other/Unknown":
        if theme_group(theme_a) == theme_group(theme_b):
            return True

    return False


def entities_are_similar(row_a, row_b):
    candidates_a = split_items(row_a.get("candidates_present", ""))
    candidates_b = split_items(row_b.get("candidates_present", ""))
    parties_a = split_items(row_a.get("parties_present", ""))
    parties_b = split_items(row_b.get("parties_present", ""))

    return bool((candidates_a & candidates_b) or (parties_a & parties_b))


def zones_are_compatible(row_a, row_b):
    va = vertical_from_screen_zone(row_a.get("dominant_screen_zone", "unknown"))
    vb = vertical_from_screen_zone(row_b.get("dominant_screen_zone", "unknown"))

    if "unknown" in {va, vb}:
        return True

    # bottom_center -> bottom_right é compatível; bottom -> middle já não é.
    return va == vb


def build_blocks_from_window_groups(window_df, id_col="block_id"):
    block_rows = []

    for block_id, group in window_df.groupby(id_col):
        group = group.sort_values("window_start_sec")
        start_sec = int(group["window_start_sec"].min())
        end_sec = int(group["window_end_sec"].max())

        first = group.iloc[0]
        start_reason = first.get("transition_reasons", "")
        if int(block_id) == 1 and id_col == "block_id":
            start_reason = "start of video"

        block_rows.append({
            id_col: int(block_id),

            "start_sec": start_sec,
            "end_sec": end_sec,
            "start_time": seconds_to_hhmmss(start_sec),
            "end_time": seconds_to_hhmmss(end_sec),
            "duration_sec": end_sec - start_sec,
            "duration_min": round((end_sec - start_sec) / 60, 2),

            # Como o OCR está a 1 FPS, segundo ≈ frame.
            # end_frame usa end_sec - 1 porque o intervalo é [start_sec, end_sec)
            "start_frame": start_sec,
            "end_frame": max(start_sec, end_sec - 1),
            "duration_frames": end_sec - start_sec,

            "n_windows": int(len(group)),

            "n_ocr_detections": int(group["n_detections"].sum()),
            "avg_confidence": group["avg_confidence"].mean(),
            "dominant_theme": weighted_dominant_theme(group),
            "themes_present": concat_unique_non_empty(group["themes_present"].tolist()),
            "candidates_present": concat_unique_non_empty(group["candidates_present"].tolist()),
            "parties_present": concat_unique_non_empty(group["parties_present"].tolist()),
            "dominant_screen_zone": mode_or_unknown(group["dominant_screen_zone"]),
            "start_transition_reason": start_reason if start_reason else "not specified",
            "max_transition_score_inside": int(group["transition_score"].max()) if "transition_score" in group else 0,
            "n_quality_warnings": int(group["quality_warning"].sum()) if "quality_warning" in group else 0,
            "example_text": top_example_texts(
                ocr_dedup[(ocr_dedup["second"] >= start_sec) & (ocr_dedup["second"] < end_sec)],
                max_items=8,
                max_chars=500,
            ),
        })

    return pd.DataFrame(block_rows)


# ------------------------------------------------------------
# 11.1 Micro-blocos OCR
# ------------------------------------------------------------
estimated_blocks = build_blocks_from_window_groups(blocks_df, id_col="block_id")

print("Micro-blocos OCR:", len(estimated_blocks))
display(
    estimated_blocks
    .sort_values("block_id")
    .set_index("block_id")
    .head(30)
)

estimated_blocks.to_csv(OUTPUT_DIR / "ocr_estimated_micro_blocks.csv", index=False)

plt.figure(figsize=(18, 4))
plt.bar(estimated_blocks["block_id"], estimated_blocks["duration_min"])
plt.title("Estimated OCR micro-block durations")
plt.xlabel("micro block_id")
plt.ylabel("duration_min")
plt.xticks(estimated_blocks["block_id"][::max(1, len(estimated_blocks)//30)])
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 11.2 Consolidação de micro-blocos em blocos/notícias mais estáveis
# ------------------------------------------------------------
estimated_blocks = estimated_blocks.sort_values("start_sec").reset_index(drop=True)

consolidated_ids = []
current_consolidated_id = 1

for i, row in estimated_blocks.iterrows():
    if i == 0:
        consolidated_ids.append(current_consolidated_id)
        continue

    prev = estimated_blocks.iloc[i - 1]

    current_is_short = row["duration_sec"] < MIN_FINAL_BLOCK_SECONDS
    prev_is_short = prev["duration_sec"] < MIN_FINAL_BLOCK_SECONDS

    similar_content = (
        themes_are_similar(prev, row) or
        entities_are_similar(prev, row)
    )

    compatible_zone = zones_are_compatible(prev, row)

    same_dominant_theme = (
        prev["dominant_theme"] == row["dominant_theme"]
        and row["dominant_theme"] != "Other/Unknown"
    )

    shared_themes = (
        len(split_items(prev["themes_present"]) & split_items(row["themes_present"])) > 0
    )

    start_reason = str(row.get("start_transition_reason", "")).lower()

    strong_theme_boundary = "strong theme change" in start_reason

    # Regra antiga:
    # junta se um dos blocos for curto e houver continuidade temática/entidades.
    short_block_merge = (
        (current_is_short or prev_is_short)
        and similar_content
        and compatible_zone
    )

    # Nova regra:
    # também junta blocos longos quando parecem claramente continuação do mesmo assunto.
    long_same_story_merge = (
        same_dominant_theme
        and shared_themes
        and compatible_zone
        and not strong_theme_boundary
    )

    should_merge = short_block_merge or long_same_story_merge

    if should_merge:
        consolidated_ids.append(current_consolidated_id)
    else:
        current_consolidated_id += 1
        consolidated_ids.append(current_consolidated_id)

estimated_blocks["consolidated_block_id"] = consolidated_ids

block_to_consolidated = dict(zip(estimated_blocks["block_id"], estimated_blocks["consolidated_block_id"]))
blocks_df["consolidated_block_id"] = blocks_df["block_id"].map(block_to_consolidated)

consolidated_blocks = build_blocks_from_window_groups(blocks_df, id_col="consolidated_block_id")
consolidated_blocks = consolidated_blocks.rename(columns={"consolidated_block_id": "block_id"})

# Guardar também que micro-blocos entraram em cada bloco consolidado.
members = (
    estimated_blocks
    .groupby("consolidated_block_id")["block_id"]
    .apply(lambda x: ", ".join(map(str, x.tolist())))
    .reset_index()
    .rename(columns={"consolidated_block_id": "block_id", "block_id": "micro_blocks"})
)
consolidated_blocks = consolidated_blocks.merge(members, on="block_id", how="left")

print("Blocos consolidados:", len(consolidated_blocks))

cols_to_show = [
    "block_id",
    "start_time",
    "end_time",
    "duration_min",
    "start_frame",
    "end_frame",
    "duration_frames",
    "n_windows",
    "dominant_theme",
    "themes_present",
    "candidates_present",
    "parties_present",
    "dominant_screen_zone",
    "micro_blocks",
    "start_transition_reason",
    "example_text"
]

cols_to_show = [c for c in cols_to_show if c in consolidated_blocks.columns]

consolidated_blocks_view = (
    consolidated_blocks[cols_to_show]
    .sort_values("block_id")
    .reset_index(drop=True)
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", 400,
    "display.width", 2000
):
    display(
    consolidated_blocks_view
    .sort_values("block_id")
    .set_index("block_id")
)

consolidated_blocks.to_csv(OUTPUT_DIR / "ocr_consolidated_news_blocks.csv", index=False)

plt.figure(figsize=(18, 4))
plt.bar(consolidated_blocks["block_id"], consolidated_blocks["duration_min"])
plt.title("Consolidated OCR-based news block durations")
plt.xlabel("consolidated block_id")
plt.ylabel("duration_min")
plt.xticks(consolidated_blocks["block_id"][::max(1, len(consolidated_blocks)//30)])
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Inspecionar um bloco específico
# ============================================================
# Escolhe:
# - BLOCK_TABLE = "micro" para ver os blocos OCR originais;
# - BLOCK_TABLE = "consolidated" para ver os blocos finais recomendados para apresentação.
# ============================================================

BLOCK_TABLE = "consolidated"
BLOCK_ID = 5         # muda aqui para o bloco que queres inspecionar

if BLOCK_TABLE == "micro":
    block_source = estimated_blocks.rename(columns={"block_id": "inspect_block_id"}).copy()
    id_col = "inspect_block_id"
elif BLOCK_TABLE == "consolidated":
    block_source = consolidated_blocks.rename(columns={"block_id": "inspect_block_id"}).copy()
    id_col = "inspect_block_id"
else:
    raise ValueError("BLOCK_TABLE deve ser 'micro' ou 'consolidated'.")

block = block_source[block_source[id_col] == BLOCK_ID]

if block.empty:
    print(f"Block {BLOCK_ID} não existe em {BLOCK_TABLE}.")
else:
    row = block.iloc[0]

    print("Block table:", BLOCK_TABLE)
    print("Block ID:", BLOCK_ID)
    print("Start time:", row["start_time"])
    print("End time:", row["end_time"])
    print("Duration:", row["duration_min"], "min")
    print("Approx start frame:", int(row["start_sec"]))
    print("Approx end frame:", int(row["end_sec"]))
    print("Dominant theme:", row["dominant_theme"])
    print("Themes present:", row["themes_present"])
    print("Candidates:", row["candidates_present"])
    print("Dominant screen zone:", row["dominant_screen_zone"])
    print("Start transition reason:", row["start_transition_reason"])
    print("Max transition score inside:", row.get("max_transition_score_inside", "NA"))
    print("N quality warnings:", row.get("n_quality_warnings", "NA"))

    if "micro_blocks" in row:
        print("Micro-blocks included:", row["micro_blocks"])

    print("\nExample OCR text:")
    print(row["example_text"])

    # Mostrar janelas internas do bloco para debug.
    internal = blocks_df[
        (blocks_df["window_start_sec"] >= row["start_sec"]) &
        (blocks_df["window_end_sec"] <= row["end_sec"])
    ].copy()

    debug_cols = [
        "time_start", "time_end",
        "dominant_theme", "themes_present", "dominant_screen_zone",
        "n_detections", "avg_confidence",
        "lexical_change", "volume_abs_change",
        "theme_changed", "strong_theme_change",
        "lexical_change_high", "volume_change_high",
        "minor_zone_change", "major_zone_change",
        "confidence_drop", "new_theme_after_unknown", "new_theme_after_unknown_persistent",
        "transition_score", "is_transition_candidate", "transition_reasons",
        "example_text",
    ]

    available_cols = [c for c in debug_cols if c in internal.columns]
    display(internal[available_cols])


## 12. Janelas e blocos interessantes para validação manual

Esta tabela é pensada para o grupo usar em validação multimodal.

Nesta versão distinguimos:

- **transition candidates**: momentos onde o OCR sugere uma mudança;
- **consolidated blocks**: blocos finais mais estáveis para apresentar como notícias/blocos estimados.

Exemplos de validação:

- verificar na transcrição se o tema corresponde;
- verificar no frame se há mudança de pivot para peça;
- verificar se há novo título, rodapé, gráfico ou imagem de apoio.


In [ ]:
validation_table = transition_candidates.copy()

if validation_table.empty:
    print("Não houve transition candidates com o threshold atual. Vou mostrar as janelas com score mais alto.")
    validation_table = window_timeline.sort_values("transition_score", ascending=False).head(15).copy()

validation_table = validation_table.sort_values("window_start_sec").copy()
validation_table["ocr_finding"] = validation_table.apply(
    lambda r: (
        f"Possible transition: {r.get('prev_theme', 'unknown')} -> {r.get('dominant_theme', 'unknown')}; "
        f"score={r.get('transition_score', '')}; reasons: {r.get('transition_reasons', '')}"
    ),
    axis=1,
)
validation_table["confidence_evidence"] = validation_table.apply(
    lambda r: (
        "quality warning: confidence drop"
        if bool(r.get("quality_warning", False))
        else ("no confidence" if pd.isna(r.get("avg_confidence", np.nan)) else f"avg confidence = {r.get('avg_confidence'):.2f}")
    ),
    axis=1,
)
validation_table["location_evidence"] = validation_table.apply(
    lambda r: (
        f"zone: {r.get('prev_zone', 'unknown')} -> {r.get('dominant_screen_zone', 'unknown')}; "
        f"major_zone_change={bool(r.get('major_zone_change', False))}"
    ),
    axis=1,
)
validation_table["validate_with_speech"] = "Check Whisper transcript around this time window."
validation_table["validate_with_visual"] = "Check frame/video around this time window for anchor, title, lower-third or graphic change."

validation_table_out = validation_table[[
    "time_start", "time_end", "transition_score", "has_strong_signal", "ocr_finding", "confidence_evidence",
    "location_evidence", "example_text", "validate_with_speech", "validate_with_visual"
]].copy()

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)

display(validation_table_out.head(30))

validation_table_out.to_csv(OUTPUT_DIR / "ocr_validation_table_for_multimodal_analysis.csv", index=False)

# Tabela de blocos consolidados para validação.
consolidated_validation = consolidated_blocks[[
    "block_id", "start_time", "end_time", "duration_min", "dominant_theme",
    "themes_present", "candidates_present", "dominant_screen_zone",
    "avg_confidence", "n_quality_warnings", "micro_blocks", "example_text"
]].copy()

consolidated_validation["validate_with_speech"] = "Check whether the transcript topic matches the OCR theme."
consolidated_validation["validate_with_visual"] = "Check whether the visual scene/layout supports this block boundary."

display(consolidated_validation.head(30))
consolidated_validation.to_csv(OUTPUT_DIR / "ocr_consolidated_blocks_validation_table.csv", index=False)


## 13. Síntese automática para discutir com a professora

Esta célula gera uma síntese curta que podes adaptar para a conversa/apresentação.


In [ ]:
n_micro_blocks = len(estimated_blocks)
n_consolidated_blocks = len(consolidated_blocks)
n_transition_candidates = len(transition_candidates)
video_duration_min = round(ocr_long["second"].max() / 60, 2) if len(ocr_long) else np.nan
bbox_ratio = ocr_filtered["bbox_available"].mean() if "bbox_available" in ocr_filtered else np.nan
avg_conf = ocr_for_segmentation["confidence"].mean() if "confidence" in ocr_for_segmentation else np.nan

main_themes = (
    consolidated_blocks["dominant_theme"]
    .value_counts()
    .head(5)
    .index
    .tolist()
)

summary_text = f'''
### Draft summary

For the single newscast OCR analysis, I used the visible on-screen text to build a temporal representation of the video.
The analysis combines three OCR dimensions: text content, confidence and screen location.

- Target file: `{target_path.name}`
- Approximate OCR-covered duration: {video_duration_min} minutes
- OCR detections after confidence filtering, full screen: {len(ocr_all)}
- OCR detections used for segmentation, region `{SEGMENTATION_REGION}`: {len(ocr_for_segmentation)}
- Detections after lower-third/segmentation deduplication: {len(ocr_dedup)}
- Segmentation vertical threshold: y2 >= {lower_third_y_min}
- Bounding box availability: {bbox_ratio:.1%}
- Average OCR confidence in segmentation region: {avg_conf:.3f} if available
- Window size: {WINDOW_SIZE_SECONDS} seconds
- Number of OCR-based transition candidates: {n_transition_candidates}
- Number of micro OCR blocks: {n_micro_blocks}
- Number of consolidated OCR-based blocks: {n_consolidated_blocks}
- Main detected themes after consolidation: {', '.join(main_themes)}

Methodological note:
For estimating news/story boundaries, the final segmentation uses OCR from the lower-third region rather than the full screen.
This region usually contains titles, subtitles and editorial lower-thirds, so it is more appropriate for detecting news topics and transitions.
The full-screen OCR is still kept for context and manual validation.

The OCR should not be interpreted as a definitive segmentation of the newscast.
Instead, it provides candidate transition points and estimated thematic blocks that should be validated with speech transcripts and visual inspection.

Micro-blocks are produced from 3-second OCR windows and can reflect small OCR/layout changes.
For estimating news/story duration, the consolidated blocks are more appropriate because they merge short adjacent OCR micro-blocks with similar themes/entities/locations.
'''

display(Markdown(summary_text))

with open(OUTPUT_DIR / "draft_summary_single_newscast_ocr.md", "w", encoding="utf-8") as f:
    f.write(summary_text)


## 12. OCR-only anchor-to-piece transition detection


In [ ]:
# ------------------------------------------------------------
# 12. OCR-only anchor/pivot → peça/notícia transition detection
# ------------------------------------------------------------

import re
import unicodedata
import numpy as np
import pandas as pd

# Parâmetros principais
LOWER_THIRD_Y_MIN = 540
FRAME_WIDTH = 1280
FRAME_HEIGHT = 720

# antes estava 60; agora reduzimos para evitar candidatos demasiado tarde
ANCHOR_SEARCH_SECONDS = 90 # aaqui era 45 n esquecer

# antes estava 9; baixamos para permitir detetar transições mais cedo
BASELINE_SECONDS = 6

ANCHOR_WINDOW_SECONDS = 3

# só confiamos em transições que aparecem cedo na notícia
EARLY_CANDIDATE_MAX_OFFSET = 35

MIN_NEW_TERMS = 2
MIN_NEW_TERMS_RATIO = 0.35

MIN_SCORE_FOR_CANDIDATE = 2
MIN_SCORE_FOR_STRONG = 3
MIN_PERSISTENCE_WINDOWS = 2

MIN_BODY_OCR_CONFIDENCE = 0.50

STUDIO_MARKERS = [
    "telejornal",
    "rtp notícias",
    "rtp noticias"
]

def basic_clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^a-z0-9À-ÿ\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# Usa a tua função se já existir; senão usa fallback simples.
CLEANER = globals().get("clean_text_pt", basic_clean_text)


if "seconds_to_hhmmss" not in globals():
    def seconds_to_hhmmss(seconds):
        seconds = int(seconds)
        h = seconds // 3600
        m = (seconds % 3600) // 60
        s = seconds % 60
        return f"{h:02d}:{m:02d}:{s:02d}"


# Detetar coluna de texto
TEXT_COL = "text" if "text" in ocr_dedup.columns else None
if TEXT_COL is None:
    raise ValueError("Não encontrei coluna 'text' em ocr_dedup.")

SECOND_COL = "second" if "second" in ocr_dedup.columns else None
if SECOND_COL is None:
    raise ValueError("Não encontrei coluna 'second' em ocr_dedup.")

required_cols = ["x1", "y1", "x2", "y2"]
missing = [c for c in required_cols if c not in ocr_dedup.columns]
if missing:
    raise ValueError(f"Faltam colunas de coordenadas em ocr_dedup: {missing}")


ocr_anchor = ocr_dedup.copy()

ocr_anchor["second"] = ocr_anchor[SECOND_COL].astype(int)
ocr_anchor["anchor_clean_text"] = ocr_anchor[TEXT_COL].apply(CLEANER)

ocr_anchor["x_center"] = (ocr_anchor["x1"] + ocr_anchor["x2"]) / 2
ocr_anchor["y_center"] = (ocr_anchor["y1"] + ocr_anchor["y2"]) / 2

# Tudo o que toca no rodapé é tratado como lower-third.
ocr_anchor["is_lower_third"] = ocr_anchor["y2"] >= LOWER_THIRD_Y_MIN

# Para esta tarefa, queremos apenas OCR fora do rodapé.
ocr_body = ocr_anchor[~ocr_anchor["is_lower_third"]].copy()

if "confidence" in ocr_body.columns:
    ocr_body = ocr_body[
        ocr_body["confidence"].isna() |
        (ocr_body["confidence"] >= MIN_BODY_OCR_CONFIDENCE)
    ].copy()


def assign_body_zone(row):
    x = row["x_center"]
    y = row["y_center"]

    vertical = "top" if y < FRAME_HEIGHT / 3 else "middle"

    if x < FRAME_WIDTH / 3:
        horizontal = "left"
    elif x < 2 * FRAME_WIDTH / 3:
        horizontal = "center"
    else:
        horizontal = "right"

    return f"{vertical}_{horizontal}"


ocr_body["body_zone"] = ocr_body.apply(assign_body_zone, axis=1)

print("OCR total:", len(ocr_anchor))
print("OCR no rodapé:", int(ocr_anchor["is_lower_third"].sum()))
print("OCR fora do rodapé:", len(ocr_body))

In [ ]:
def text_to_tokens(text):
    text = CLEANER(text)
    tokens = set()

    for token in text.split():
        if len(token) < 3:
            continue
        if token.isdigit():
            continue
        tokens.add(token)

    return tokens


STUDIO_MARKERS_CLEAN = [CLEANER(x) for x in STUDIO_MARKERS]


def has_studio_marker(text):
    clean = CLEANER(text)
    return any(marker in clean for marker in STUDIO_MARKERS_CLEAN if marker)


def join_unique_texts(values, max_items=6, max_chars=300):
    out = []

    for v in values:
        if pd.isna(v):
            continue

        s = str(v).strip()
        if not s:
            continue

        if s not in out:
            out.append(s)

        if len(out) >= max_items:
            break

    joined = " | ".join(out)

    if len(joined) > max_chars:
        joined = joined[:max_chars] + "..."

    return joined


def mode_or_none(values):
    s = pd.Series(values).dropna()
    s = s[s.astype(str) != "none"]

    if len(s) == 0:
        return "none"

    return s.mode().iloc[0]


def build_body_windows_for_block(block_row):
    block_id = int(block_row["block_id"])
    block_start = int(block_row["start_sec"])
    block_end = int(block_row["end_sec"])

    search_start = block_start
    search_end = min(block_end, block_start + ANCHOR_SEARCH_SECONDS)

    rows = []

    for ws in range(search_start, search_end, ANCHOR_WINDOW_SECONDS):
        we = min(ws + ANCHOR_WINDOW_SECONDS, search_end)

        g = ocr_body[
            (ocr_body["second"] >= ws) &
            (ocr_body["second"] < we)
        ].copy()

        clean_text = " ".join(g["anchor_clean_text"].dropna().astype(str).tolist())
        tokens = text_to_tokens(clean_text)

        rows.append({
            "block_id": block_id,
            "window_start_sec": ws,
            "window_end_sec": we,
            "window_start_time": seconds_to_hhmmss(ws),
            "window_end_time": seconds_to_hhmmss(we),
            "start_frame": ws,
            "end_frame": max(ws, we - 1),

            "n_body_ocr": int(len(g)),
            "n_body_tokens": int(len(tokens)),
            "body_tokens": tokens,
            "body_text_clean": clean_text,
            "body_example_text": join_unique_texts(g[TEXT_COL].tolist(), max_items=6, max_chars=300),
            "dominant_body_zone": mode_or_none(g["body_zone"]) if len(g) else "none",
            "has_studio_marker": has_studio_marker(clean_text),
        })

    return pd.DataFrame(rows)

In [ ]:
def score_anchor_to_piece_windows(windows_df, block_row):
    if windows_df.empty:
        return windows_df

    block_start = int(block_row["start_sec"])
    baseline_end = block_start + BASELINE_SECONDS

    baseline = windows_df[windows_df["window_start_sec"] < baseline_end].copy()

    if baseline.empty:
        baseline = windows_df.head(1).copy()

    baseline_tokens = set()
    for tokens in baseline["body_tokens"]:
        baseline_tokens |= tokens

    baseline_avg_ocr = baseline["n_body_ocr"].mean()
    baseline_zone = mode_or_none(baseline["dominant_body_zone"])
    baseline_has_studio = bool(baseline["has_studio_marker"].any())
    baseline_text = join_unique_texts(
        baseline["body_example_text"].tolist(),
        max_items=8,
        max_chars=400
    )

    scored_rows = []

    for _, row in windows_df.iterrows():
        tokens = row["body_tokens"]
        new_terms = tokens - baseline_tokens

        union = tokens | baseline_tokens
        if len(union) == 0:
            jaccard_distance = 0.0
        else:
            jaccard_distance = 1 - (len(tokens & baseline_tokens) / len(union))

        new_terms_ratio = len(new_terms) / max(1, len(tokens))

        offset_from_news_start_sec = int(row["window_start_sec"] - block_start)

        body_appeared_after_empty_baseline = (
            baseline_avg_ocr < 0.5 and row["n_body_ocr"] >= 1
        )

        body_volume_jump = (
            row["n_body_ocr"] >= max(2, baseline_avg_ocr + 2)
        )

        new_terms_signal = (
            len(new_terms) >= MIN_NEW_TERMS and
            new_terms_ratio >= MIN_NEW_TERMS_RATIO
        )

        lexical_shift_signal = (
            jaccard_distance >= 0.70 and
            row["n_body_tokens"] >= 2
        )

        zone_changed = (
            baseline_zone != "none" and
            row["dominant_body_zone"] != "none" and
            row["dominant_body_zone"] != baseline_zone
        )

        studio_marker_disappeared = (
            baseline_has_studio and
            not row["has_studio_marker"]
        )

        score = 0
        evidence = []

        if body_appeared_after_empty_baseline:
            score += 1
            evidence.append("body_ocr_appeared_after_empty_baseline")

        if body_volume_jump:
            score += 1
            evidence.append("body_ocr_volume_jump")

        if new_terms_signal:
            score += 2
            evidence.append("new_body_terms")

        if lexical_shift_signal:
            score += 1
            evidence.append("lexical_shift_from_baseline")

        if zone_changed:
            score += 1
            evidence.append("body_zone_changed")

        # damos mais peso a isto porque é dos sinais mais úteis para pivot → peça
        if studio_marker_disappeared:
            score += 2
            evidence.append("studio_marker_disappeared")

        scored = row.to_dict()
        scored.update({
            "offset_from_news_start_sec": offset_from_news_start_sec,
            "baseline_end_sec": baseline_end,
            "baseline_body_avg_ocr": round(float(baseline_avg_ocr), 2),
            "baseline_body_zone": baseline_zone,
            "baseline_has_studio_marker": baseline_has_studio,
            "baseline_body_text": baseline_text,

            "n_new_terms": int(len(new_terms)),
            "new_terms_ratio": round(float(new_terms_ratio), 3),
            "jaccard_distance_from_baseline": round(float(jaccard_distance), 3),
            "new_terms": ", ".join(sorted(list(new_terms))[:12]),

            "anchor_to_piece_score": int(score),
            "anchor_to_piece_evidence": ", ".join(evidence) if evidence else "none",
            "is_after_baseline": row["window_start_sec"] >= baseline_end,
            "is_late_window": offset_from_news_start_sec > EARLY_CANDIDATE_MAX_OFFSET,
        })

        scored_rows.append(scored)

    scored_df = pd.DataFrame(scored_rows)

    signal = (
        scored_df["is_after_baseline"] &
        (scored_df["anchor_to_piece_score"] >= MIN_SCORE_FOR_CANDIDATE)
    )

    persistence = []

    for i in range(len(scored_df)):
        count = 0
        for j in range(i, len(scored_df)):
            if bool(signal.iloc[j]):
                count += 1
            else:
                break
        persistence.append(count)

    scored_df["persistence_windows"] = persistence

    scored_df["is_persistent_candidate"] = (
        signal &
        (scored_df["persistence_windows"] >= MIN_PERSISTENCE_WINDOWS)
    )

    return scored_df

In [ ]:
anchor_candidate_rows = []
all_anchor_windows = []

for _, block_row in consolidated_blocks.sort_values("start_sec").iterrows():
    block_id = int(block_row["block_id"])
    block_start = int(block_row["start_sec"])
    block_end = int(block_row["end_sec"])

    windows = build_body_windows_for_block(block_row)
    windows = score_anchor_to_piece_windows(windows, block_row)

    if not windows.empty:
        all_anchor_windows.append(windows)

    candidate = None
    status = "not_detected_by_ocr"
    confidence = "none"

    if not windows.empty:
        # 1) Melhor caso: candidato persistente e cedo
        persistent_early = windows[
            windows["is_persistent_candidate"] &
            (windows["offset_from_news_start_sec"] <= EARLY_CANDIDATE_MAX_OFFSET)
        ].copy()

        if not persistent_early.empty:
            candidate = persistent_early.sort_values("window_start_sec").iloc[0]
            status = "candidate_detected"

            if candidate["anchor_to_piece_score"] >= MIN_SCORE_FOR_STRONG:
                confidence = "high"
            else:
                confidence = "medium"

        else:
            # 2) Sinal cedo, mas sem persistência
            weak_early = windows[
                windows["is_after_baseline"] &
                (windows["anchor_to_piece_score"] >= MIN_SCORE_FOR_CANDIDATE) &
                (windows["offset_from_news_start_sec"] <= EARLY_CANDIDATE_MAX_OFFSET)
            ].copy()

            if not weak_early.empty:
                candidate = weak_early.sort_values("window_start_sec").iloc[0]
                status = "weak_candidate_no_persistence"
                confidence = "low"

            else:
                # 3) Há sinal, mas tarde demais para confiar como pivot → peça
                late_change = windows[
                    windows["is_after_baseline"] &
                    (windows["anchor_to_piece_score"] >= MIN_SCORE_FOR_CANDIDATE) &
                    (windows["offset_from_news_start_sec"] > EARLY_CANDIDATE_MAX_OFFSET)
                ].copy()

                if not late_change.empty:
                    candidate = late_change.sort_values("window_start_sec").iloc[0]
                    status = "late_body_ocr_change"
                    confidence = "low"

    if candidate is not None:
        candidate_sec = int(candidate["window_start_sec"])
        candidate_time = seconds_to_hhmmss(candidate_sec)
        candidate_frame = candidate_sec

        score = int(candidate["anchor_to_piece_score"])
        evidence = candidate["anchor_to_piece_evidence"]
        body_text = candidate["body_example_text"]
        new_terms = candidate["new_terms"]
        persistence_windows = int(candidate["persistence_windows"])
        offset_sec = int(candidate["offset_from_news_start_sec"])

        # Só consideramos pivot_to_piece final se for cedo.
        if status in ["candidate_detected", "weak_candidate_no_persistence"]:
            pivot_to_piece_time = candidate_time
            pivot_to_piece_sec = candidate_sec
            pivot_to_piece_frame = candidate_frame
            late_change_time = None
            late_change_frame = None
        else:
            pivot_to_piece_time = None
            pivot_to_piece_sec = None
            pivot_to_piece_frame = None
            late_change_time = candidate_time
            late_change_frame = candidate_frame

    else:
        candidate_sec = None
        candidate_time = None
        candidate_frame = None
        score = 0
        evidence = "no_relevant_body_ocr_change"
        body_text = ""
        new_terms = ""
        persistence_windows = 0
        offset_sec = None

        pivot_to_piece_time = None
        pivot_to_piece_sec = None
        pivot_to_piece_frame = None
        late_change_time = None
        late_change_frame = None

    anchor_candidate_rows.append({
        "block_id": block_id,

        "news_start_time": seconds_to_hhmmss(block_start),
        "news_end_time": seconds_to_hhmmss(block_end),
        "news_duration_min": round((block_end - block_start) / 60, 2),

        "news_start_frame": block_start,
        "news_end_frame": max(block_start, block_end - 1),

        "candidate_status": status,
        "confidence": confidence,

        "pivot_to_piece_time": pivot_to_piece_time,
        "pivot_to_piece_sec": pivot_to_piece_sec,
        "pivot_to_piece_frame": pivot_to_piece_frame,

        "late_change_time": late_change_time,
        "late_change_frame": late_change_frame,

        "candidate_offset_sec": offset_sec,
        "score": score,
        "persistence_windows": persistence_windows,
        "evidence": evidence,
        "new_body_terms": new_terms,
        "candidate_body_ocr": body_text,

        "dominant_theme": block_row.get("dominant_theme", "unknown"),
        "micro_blocks": block_row.get("micro_blocks", ""),
    })

anchor_to_piece_candidates = pd.DataFrame(anchor_candidate_rows)

if all_anchor_windows:
    anchor_to_piece_windows_all = pd.concat(all_anchor_windows, ignore_index=True)
else:
    anchor_to_piece_windows_all = pd.DataFrame()

print("Blocos analisados:", len(anchor_to_piece_candidates))
print(anchor_to_piece_candidates["candidate_status"].value_counts(dropna=False))

In [ ]:
cols_to_show = [
    "block_id",
    "news_start_time",
    "news_end_time",
    "news_duration_min",
    "news_start_frame",
    "news_end_frame",

    "candidate_status",
    "confidence",
    "pivot_to_piece_time",
    "pivot_to_piece_frame",
    "candidate_offset_sec",

    "late_change_time",
    "late_change_frame",

    "score",
    "persistence_windows",
    "evidence",
    "new_body_terms",
    "candidate_body_ocr",

    "dominant_theme",
    "micro_blocks",
]

cols_to_show = [c for c in cols_to_show if c in anchor_to_piece_candidates.columns]

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", 400,
    "display.width", 2000
):
    display(anchor_to_piece_candidates[cols_to_show])

In [ ]:
try:
    anchor_to_piece_candidates.to_csv(
        OUTPUT_DIR / "ocr_anchor_to_piece_candidates.csv",
        index=False
    )

    anchor_to_piece_windows_all.to_csv(
        OUTPUT_DIR / "ocr_anchor_to_piece_windows_debug.csv",
        index=False
    )

    print("Ficheiros guardados em:", OUTPUT_DIR)

except NameError:
    print("OUTPUT_DIR não existe. Resultados não foram guardados em CSV.")

In [ ]:
debug_cols = [
    "block_id",
    "window_start_time",
    "window_end_time",
    "start_frame",
    "end_frame",
    "n_body_ocr",
    "dominant_body_zone",
    "has_studio_marker",
    "anchor_to_piece_score",
    "persistence_windows",
    "is_persistent_candidate",
    "anchor_to_piece_evidence",
    "new_terms",
    "body_example_text",
]

debug_cols = [c for c in debug_cols if c in anchor_to_piece_windows_all.columns]

for block_id in sorted(anchor_to_piece_windows_all["block_id"].unique()):
    print("=" * 120)
    print(f"BLOCO {block_id}")
    print("=" * 120)

    tmp = (
        anchor_to_piece_windows_all[
            anchor_to_piece_windows_all["block_id"] == block_id
        ]
        .sort_values("window_start_sec")
        .copy()
    )

    with pd.option_context(
        "display.max_rows", None,
        "display.max_columns", None,
        "display.max_colwidth", 500,
        "display.width", 2000
    ):
        display(tmp[debug_cols])

## 16 ocr por frame


In [ ]:
# ============================================================
# Inspecionar OCR de múltiplos frames específicos
# ============================================================

FRAMES_TO_INSPECT = [
    134
]

USE_FILTERED = False          # False = todos os OCR; True = só OCR após filtro de confiança
USE_NEAREST_IF_EMPTY = True   # True = se não houver OCR no frame, mostra o frame mais próximo com OCR
SAVE_CSV = True               # Guarda resultado em CSV

# Escolher fonte
if USE_FILTERED:
    source_df = ocr_filtered.copy()
    source_label = "ocr_filtered"
else:
    source_df = ocr_long.copy()
    source_label = "ocr_long"

# Garantir que temos localização/zona
if "screen_zone" not in source_df.columns:
    if "add_bbox_features" not in globals():
        raise NameError("Corre primeiro a célula que define add_bbox_features().")
    source_df = add_bbox_features(source_df)

# Criar versão inteira do frame para evitar problemas de comparação
source_df["frame_int"] = source_df["frame"].astype(int)

# Funções auxiliares
def fmt_coord(x):
    if pd.isna(x):
        return ""
    if abs(float(x)) <= 2:
        return f"{float(x):.3f}"
    return f"{float(x):.0f}"

def fmt_conf(x):
    if pd.isna(x):
        return "no confidence"
    return f"{float(x):.3f}"

def sec_to_time(s):
    if pd.isna(s):
        return ""
    s = int(s)
    m = s // 60
    sec = s % 60
    return f"{m:02d}:{sec:02d}"

all_frame_dfs = []
summary_rows = []

available_frames = np.array(sorted(source_df["frame_int"].dropna().astype(int).unique()))

if len(available_frames) == 0:
    raise ValueError("Não há frames disponíveis no OCR.")

for requested_frame in FRAMES_TO_INSPECT:
    requested_frame = int(requested_frame)

    # Procurar frame exato
    frame_df = source_df[source_df["frame_int"] == requested_frame].copy()

    # Se não houver OCR nesse frame, usar frame mais próximo com OCR
    if frame_df.empty and USE_NEAREST_IF_EMPTY:
        nearest_frame = int(available_frames[np.argmin(np.abs(available_frames - requested_frame))])
        frame_df = source_df[source_df["frame_int"] == nearest_frame].copy()
        inspected_frame = nearest_frame
        match_type = "nearest"
    elif frame_df.empty:
        inspected_frame = requested_frame
        match_type = "empty"
    else:
        inspected_frame = requested_frame
        match_type = "exact"

    if frame_df.empty:
        summary_rows.append({
            "requested_frame": requested_frame,
            "inspected_frame": inspected_frame,
            "match_type": match_type,
            "n_ocr": 0,
            "avg_confidence": np.nan
        })
        continue

    # Criar colunas legíveis
    frame_df["requested_frame"] = requested_frame
    frame_df["inspected_frame"] = inspected_frame
    frame_df["match_type"] = match_type

    frame_df["time"] = frame_df["second"].apply(sec_to_time)
    frame_df["confidence_display"] = frame_df["confidence"].apply(fmt_conf)

    frame_df["bbox"] = frame_df.apply(
        lambda r: f"({fmt_coord(r['x1'])}, {fmt_coord(r['y1'])}) → ({fmt_coord(r['x2'])}, {fmt_coord(r['y2'])})",
        axis=1
    )

    frame_df["center"] = frame_df.apply(
        lambda r: f"({fmt_coord(r['x_center'])}, {fmt_coord(r['y_center'])})",
        axis=1
    )

    # Ordenar como aparece no ecrã: de cima para baixo, esquerda para direita
    frame_df = frame_df.sort_values(
        ["requested_frame", "y1", "x1", "confidence"],
        ascending=[True, True, True, False]
    )

    all_frame_dfs.append(frame_df)

    summary_rows.append({
        "requested_frame": requested_frame,
        "inspected_frame": inspected_frame,
        "match_type": match_type,
        "n_ocr": len(frame_df),
        "avg_confidence": frame_df["confidence"].mean()
    })

# Resumo
summary_df = pd.DataFrame(summary_rows)

print(f"Fonte usada: {source_label}")
print(f"Número de frames pedidos: {len(FRAMES_TO_INSPECT)}")
print(f"Número total de deteções OCR encontradas: {sum(summary_df['n_ocr'])}")

display(summary_df)

# Tabela final com todos os OCR
if len(all_frame_dfs) == 0:
    print("Não houve OCR para nenhum dos frames pedidos.")
else:
    multi_frame_ocr = pd.concat(all_frame_dfs, ignore_index=True)

    cols_to_show = [
        "requested_frame",
        "inspected_frame",
        "match_type",
        "time",
        "text",
        "clean_text",
        "confidence_display",
        "bbox",
        "center",
        "zone_vertical",
        "zone_horizontal",
        "screen_zone",
    ]

    pd.set_option("display.max_colwidth", None)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 3000)

    display(
        multi_frame_ocr[cols_to_show].style.set_properties(
            subset=["text", "clean_text", "bbox", "center", "screen_zone"],
            **{
                "white-space": "normal",
                "text-align": "left",
                "min-width": "120px",
                "max-width": "500px",
            }
        )
    )

    if SAVE_CSV:
        out_path = OUTPUT_DIR / "ocr_inspection_multiple_frames.csv"
        multi_frame_ocr[cols_to_show].to_csv(out_path, index=False)
        print(f"CSV guardado em: {out_path}")

## 14. Outputs gerados

Ficheiros guardados em `outputs_ocr_04_single_newscast/`:

- `ocr_confidence_by_minute.csv`
- `ocr_zone_counts.csv`
- `ocr_zone_by_minute.csv`
- `ocr_segmentation_region_used.csv`
- `ocr_segmentation_region_volume_compare.csv`
- `ocr_single_newscast_lower_third_dedup.csv`
- `ocr_single_newscast_long_dedup.csv`
- `ocr_window_topic_timeline.csv`
- `ocr_theme_timeline_long.csv`
- `ocr_window_topic_timeline_with_lexical_change.csv`
- `ocr_window_timeline_with_transition_score.csv`
- `ocr_transition_candidates.csv`
- `ocr_estimated_micro_blocks.csv`
- `ocr_consolidated_news_blocks.csv`
- `ocr_validation_table_for_multimodal_analysis.csv`
- `ocr_consolidated_blocks_validation_table.csv`
- `draft_summary_single_newscast_ocr.md`

Outputs principais para mostrar ao grupo/professora:

1. **Timeline de qualidade OCR** — volume/confiança por minuto no OCR completo.
2. **Análise de localização** — zonas onde o texto aparece.
3. **Região usada para segmentação** — comparação OCR completo vs lower-third.
4. **Timeline temática** — temas dominantes por janela temporal, calculados a partir do lower-third.
5. **Pontos candidatos a transição** — baseados em score OCR ponderado.
6. **Micro-blocos OCR** — úteis para debug e localização fina de mudanças.
7. **Blocos/notícias consolidados** — melhor output para contar blocos e estimar duração.
8. **Tabela de validação multimodal** — o que verificar com speech/visual.

### Nota metodológica importante

Nesta versão, a segmentação principal usa OCR que intersecta a zona inferior do ecrã (`y2 >= 480` em vídeo 720p), pois é onde aparecem títulos e subtítulos editoriais.

Como a análise usa janelas de 3 segundos, os micro-blocos não devem ser interpretados automaticamente como notícias reais.
Os blocos consolidados juntam micro-blocos consecutivos quando mantêm continuidade temática, entidades semelhantes ou localização compatível.

O OCR completo continua disponível para contexto e validação manual.
